# Snaps ↔ earthquakes: catalog dictionary & comparisons

Side-by-side checks of simulation catalogs (**snaps / cascades / avalanches**)
against real earthquake catalogs (**JMA**, **Hauksson**).

| Sim level | Closest quake analogue | Size proxy |
|---|---|---|
| snaps | asperity / micro-event | unit (`magnitude=0`); optional `surface_area` |
| cascades | single earthquake | `log10(n_swaps)` |
| avalanches | swarm / sequence | `log10(n_swaps)`, also `n_cascades` |

Earthquake catalogs use instrumental magnitude. Absolute units are **not**
commensurate — compare shapes and clustering patterns after rescaling time
to `t − t₀`.

### Reading order

1. **Load** sim + quake catalogs (§1–2), then shared helpers (§3)
2. **Sizes** — magnitude / size PDFs (§4)
3. **Temporal** structure — $dt$ vs time, waiting times, Omori, $C_v$, rates (§5)
4. **Rudyak age windows** — Fig. 2 recreation; defines windows reused below (§6)
5. **Space** — spacetime panels, map, box-counting fractal (§7–8)
6. **Clustering tests** — Zaliapin $\eta^\ast$ / $(T,R)$ (§9), Yang first-return (§10)
7. **Appendix** — experimental fixed-size cascades* / avalanches*

### Dual time presentation

Every **temporal** diagnostic is shown twice, in this order:

1. **Time domain** — native occurrence times (sim time; calendar seconds for quakes).
2. **$t^* = \ln(t)$** — simulation times mapped to $t^*=\ln(t)$ ($t>0$); waiting times
   become $\Delta\ln t$. **JMA / Hauksson stay on calendar time** in that second pass.

Spatial-only figures (size PDFs, box-counting, map view) appear once.

### Rudyak age windows

§6 recreates Rudyak et al. (2025) Fig. 2 for snaps with half-decade windows
$t \in [10^{k}, 10^{k+0.5})$. The same windows feed per-age fractal / occupancy (§8),
Zaliapin (§9.2), and Yang (§10.2).

### Catalog windows

Simulation loading matches `notebooks/dt_vs_time.ipynb`. Quake windows:

- **JMA** — 2009–2013 (`[2009-01-01, 2014-01-01)`), Ogata–Zhuang (2006) offshore Tohoku box **36–42°N, 141–145°E** (no depth cut)
- **Hauksson** — 2017–2021 (`[2017-01-01, 2022-01-01)`)


In [1]:
from __future__ import annotations

import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import skimage
from plotly.subplots import make_subplots
from skimage import measure


def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "pyproject.toml").is_file() and (base / "crumpling").is_dir():
            return base
    raise FileNotFoundError(f"Could not locate the repository above {here}")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from crumpling import catalogs
from crumpling import config
from crumpling import pipeline

print(f"python={sys.executable}")
print(f"ROOT={ROOT}")
print(f"skimage={skimage.__version__}")


python=/home/neriberman/miniconda3/envs/crumpling/bin/python
ROOT=/home/neriberman/REPOS/crumpling
skimage=0.26.0


## 1. Load simulation catalogs

Latest `*__swaps*.txt` under `data/raw/` → pipeline → `EventCatalogs`
(snaps, cascades, avalanches).


In [2]:
swap_candidates = sorted(
    (ROOT / "data/raw").glob("**/N*__swaps*.txt"),
    key=lambda path: path.stat().st_mtime,
)
if not swap_candidates:
    raise FileNotFoundError("No swaps files found under data/raw")

DATA = swap_candidates[-1]
CONFIG = ROOT / "configs/trial32/swapstat.params.ini"

settings = config.load_config(CONFIG)
result = pipeline.analyze_file(DATA, settings)
bundle = catalogs.build_catalogs(result)

print(type(bundle).__name__)
print("source:", bundle.source)
print(
    f"snaps={len(bundle.snaps)}  "
    f"cascades={len(bundle.cascades)}  "
    f"avalanches={len(bundle.avalanches)}"
)


EventCatalogs
source: /home/neriberman/REPOS/crumpling/data/raw/trial32/N40K-compress-LJ10-T0.001-F0.61-damp4.0_stage1_trial32_dt10-dr400__swaps.txt
snaps=3242  cascades=892  avalanches=225


## 2. Load JMA & Hauksson

Calendar-time earthquake catalogs with optional mainshock origins $t_0$
(Tohoku / Ridgecrest), matching `dt_vs_time.ipynb`.


In [3]:
T0_EPS = 1e-3  # seconds before mainshock origin (matches dt_vs_time.ipynb)

# Ogata & Zhuang (2006) Region A: offshore east coast of Tohoku.
# No depth cut — keep all hypocenters inside the lat/lon box.
JMA_LAT = (36.0, 42.0)
JMA_LON = (141.0, 145.0)


def load_quake_window(
    path: pathlib.Path,
    *,
    usecols: list[str],
    label: str,
    t_start: float,
    t_end: float,
    lat_range: tuple[float, float] | None = None,
    lon_range: tuple[float, float] | None = None,
) -> pd.DataFrame:
    frame = pd.read_csv(path, usecols=usecols)
    mask = (frame["time"] >= t_start) & (frame["time"] < t_end)
    if lat_range is not None:
        lat0, lat1 = lat_range
        mask &= (frame["latitude"] >= lat0) & (frame["latitude"] <= lat1)
    if lon_range is not None:
        lon0, lon1 = lon_range
        mask &= (frame["longitude"] >= lon0) & (frame["longitude"] <= lon1)
    frame = frame.loc[mask].copy()
    frame = frame.sort_values("time", kind="mergesort").reset_index(drop=True)
    frame["dt"] = frame["time"].diff()
    frame["datetime"] = pd.to_datetime(frame["time"], unit="s")
    print(
        f"{label}: {len(frame):,} events  "
        f"({frame['datetime'].iloc[0]} → {frame['datetime'].iloc[-1]})"
    )
    print(f"  median dt = {frame['dt'].median():.3f} s")
    return frame


jma_start = pd.Timestamp("2009-01-01").timestamp()
jma_end = pd.Timestamp("2014-01-01").timestamp()
hauk_start = pd.Timestamp("2017-01-01").timestamp()
hauk_end = pd.Timestamp("2022-01-01").timestamp()

jma = load_quake_window(
    ROOT / "data/ingested/jma.csv",
    usecols=["time", "magnitude", "latitude", "longitude", "depth", "x_utm", "y_utm"],
    label=f"JMA 2009–2013 Ogata box {JMA_LAT[0]:.0f}–{JMA_LAT[1]:.0f}N, {JMA_LON[0]:.0f}–{JMA_LON[1]:.0f}E",
    t_start=jma_start,
    t_end=jma_end,
    lat_range=JMA_LAT,
    lon_range=JMA_LON,
)
hauksson = load_quake_window(
    ROOT / "data/ingested/hauksson.csv",
    usecols=["time", "magnitude", "latitude", "longitude", "depth", "x_utm", "y_utm"],
    label="Hauksson 2017–2021",
    t_start=hauk_start,
    t_end=hauk_end,
)

# Optional origin markers (same mainshocks as dt_vs_time.ipynb)
tohoku_day0 = pd.Timestamp("2011-03-11").timestamp()
tohoku_day1 = pd.Timestamp("2011-03-12").timestamp()
tohoku = (
    jma.loc[(jma["time"] >= tohoku_day0) & (jma["time"] < tohoku_day1)]
    .sort_values("magnitude", ascending=False)
    .iloc[0]
)
jma_t0 = float(tohoku["time"]) - T0_EPS

ridge_day0 = pd.Timestamp("2019-07-06").timestamp()
ridge_day1 = pd.Timestamp("2019-07-07").timestamp()
ridgecrest = (
    hauksson.loc[
        (hauksson["time"] >= ridge_day0) & (hauksson["time"] < ridge_day1)
    ]
    .sort_values("magnitude", ascending=False)
    .iloc[0]
)
hauksson_t0 = float(ridgecrest["time"]) - T0_EPS

print(f"JMA t₀ (Tohoku − ε)      → {pd.to_datetime(jma_t0, unit='s')}")
print(f"Hauksson t₀ (Ridgecrest − ε) → {pd.to_datetime(hauksson_t0, unit='s')}")

CATALOGS: dict[str, pd.DataFrame] = {
    "snaps": bundle.snaps,
    "cascades": bundle.cascades,
    "avalanches": bundle.avalanches,
    "JMA": jma,
    "Hauksson": hauksson,
}
CATALOG_T0 = {
    "snaps": None,
    "cascades": None,
    "avalanches": None,
    "JMA": jma_t0,
    "Hauksson": hauksson_t0,
}
for name, frame in CATALOGS.items():
    print(f"{name:12s} n={len(frame):>8,}")


JMA 2009–2013 Ogata box 36–42N, 141–145E: 154,208 events  (2009-01-01 00:53:25.559999943 → 2013-12-31 23:38:47.549999952)
  median dt = 441.560 s
Hauksson 2017–2021: 60,943 events  (2017-01-01 00:04:55 → 2021-12-31 23:38:36)
  median dt = 866.000 s
JMA t₀ (Tohoku − ε)      → 2011-03-11 05:46:18.118999958
Hauksson t₀ (Ridgecrest − ε) → 2019-07-06 03:19:51.999000072
snaps        n=   3,242
cascades     n=     892
avalanches   n=     225
JMA          n= 154,208
Hauksson     n=  60,943


## 3. Conventions & helpers

Shared utilities: colors, empirical PDFs, Omori / $C_v$ / rate helpers,
spatial column selection, downsampling, and dual-time
`with_log_time` / `catalogs_view` / `CATALOGS_LN`.


In [4]:
def dt_vs_relative_time(
    frame: pd.DataFrame,
    *,
    t0: float | None = None,
    max_points: int | None = None,
) -> pd.DataFrame:
    series = frame.loc[:, ["time", "dt"]].dropna(subset=["time"]).copy()
    if t0 is None:
        t0 = float(series["time"].iloc[0]) if len(series) else 0.0
    series["t"] = series["time"] - float(t0)
    series = series.loc[(series["t"] > 0) & (series["dt"] > 0), ["t", "dt"]]
    if max_points is not None and len(series) > max_points:
        step = int(np.ceil(len(series) / max_points))
        series = series.iloc[::step]
    return series


def magnitude_series(name: str, frame: pd.DataFrame) -> pd.Series:
    # Size proxy used for GR / Omori comparisons.
    if name == "snaps":
        # Unit events: show surface_area on a log10 scale as a stand-in size.
        area = frame["surface_area"].astype(float)
        positive = area > 0
        out = pd.Series(np.nan, index=frame.index, dtype=float)
        out.loc[positive] = np.log10(area.loc[positive])
        return out
    return frame["magnitude"].astype(float)


def empirical_pdf(
    values: np.ndarray,
    *,
    bins: int = 40,
    log: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if log:
        values = values[values > 0]
        if len(values) < 2:
            return np.array([]), np.array([])
        edges = np.logspace(np.log10(values.min()), np.log10(values.max()), bins + 1)
    else:
        if len(values) < 2:
            return np.array([]), np.array([])
        edges = np.linspace(values.min(), values.max(), bins + 1)
    counts, edges = np.histogram(values, bins=edges)
    widths = np.diff(edges)
    centers = np.sqrt(edges[:-1] * edges[1:]) if log else 0.5 * (edges[:-1] + edges[1:])
    pdf = counts / (counts.sum() * widths)
    mask = counts > 0
    return centers[mask], pdf[mask]


def omori_decay(
    times: np.ndarray,
    magnitudes: np.ndarray,
    *,
    m_main: float,
    t_max: float,
    n_bins: int = 25,
    max_main: int | None = None,
    rng: np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray, int]:
    # Stacked rate of later events vs time since mainshocks (M >= m_main).
    # Uses searchsorted on sorted times (O(n_main log N)). Optional max_main
    # randomly subsamples mainshocks for large catalogs; rate stays per-main.
    times = np.asarray(times, dtype=float)
    magnitudes = np.asarray(magnitudes, dtype=float)
    order = np.argsort(times, kind="mergesort")
    times = times[order]
    magnitudes = magnitudes[order]
    main = np.flatnonzero(np.isfinite(magnitudes) & (magnitudes >= m_main))
    if len(main) == 0:
        return np.array([]), np.array([]), 0
    if max_main is not None and len(main) > max_main:
        rng = np.random.default_rng() if rng is None else rng
        main = np.sort(rng.choice(main, size=max_main, replace=False))

    # Probe a few windows for the smallest positive delay (sets log-bin floor).
    probe = main[: min(len(main), 64)]
    dt_min = np.inf
    for index in probe:
        j = index + 1
        if j < len(times):
            dt = times[j] - times[index]
            if 0.0 < dt <= t_max:
                dt_min = min(dt_min, dt)
    if not np.isfinite(dt_min):
        return np.array([]), np.array([]), int(len(main))

    edges = np.logspace(np.log10(dt_min), np.log10(t_max), n_bins + 1)
    counts = np.zeros(n_bins, dtype=np.int64)
    for index in main:
        t0 = times[index]
        left = index + 1
        right = np.searchsorted(times, t0 + t_max, side="right")
        if right <= left:
            continue
        # Bin delays without materializing a giant concatenated array.
        hist, _ = np.histogram(times[left:right] - t0, bins=edges)
        counts += hist

    widths = np.diff(edges)
    centers = np.sqrt(edges[:-1] * edges[1:])
    rate = counts / (len(main) * widths)
    mask = counts > 0
    return centers[mask], rate[mask], int(len(main))



def interevent_cv(dt: np.ndarray) -> float:
    # C_v = σ_Δt / μ_Δt for positive finite inter-event times.
    dt = np.asarray(dt, dtype=float)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 2:
        return float("nan")
    mu = float(np.mean(dt))
    if mu <= 0.0:
        return float("nan")
    return float(np.std(dt, ddof=1) / mu)


def cv_vs_size_threshold(
    times: np.ndarray,
    sizes: np.ndarray,
    thresholds: np.ndarray,
    *,
    min_events: int = 50,
) -> pd.DataFrame:
    # Waiting-time C_v for events with size >= threshold (mean Δt grows with cut).
    times = np.asarray(times, dtype=float)
    sizes = np.asarray(sizes, dtype=float)
    rows: list[dict[str, float | int]] = []
    for thr in thresholds:
        keep = np.isfinite(sizes) & np.isfinite(times) & (sizes >= float(thr))
        t = np.sort(times[keep])
        if len(t) < min_events:
            continue
        dt = np.diff(t)
        dt = dt[dt > 0]
        if len(dt) < 2:
            continue
        rows.append(
            {
                "threshold": float(thr),
                "n_events": int(len(t)),
                "mean_dt": float(np.mean(dt)),
                "cv": interevent_cv(dt),
            }
        )
    return pd.DataFrame(rows)


def cv_time_series(
    times: np.ndarray,
    dt: np.ndarray,
    *,
    window: int,
) -> tuple[np.ndarray, np.ndarray]:
    # Rolling C_v of consecutive waiting times, stamped at the window end time.
    times = np.asarray(times, dtype=float)
    dt = np.asarray(dt, dtype=float)
    keep = np.isfinite(times) & np.isfinite(dt) & (dt > 0)
    times = times[keep]
    dt = dt[keep]
    if len(dt) < window:
        return np.array([]), np.array([])
    kernel = np.ones(window, dtype=float)
    means = np.convolve(dt, kernel, mode="valid") / window
    means_sq = np.convolve(dt * dt, kernel, mode="valid") / window
    var = means_sq - means * means
    var = var * (window / (window - 1))
    cv = np.sqrt(np.maximum(var, 0.0)) / means
    t_end = times[window - 1 :]
    ok = np.isfinite(cv) & (means > 0)
    return t_end[ok], cv[ok]


def downsample(frame: pd.DataFrame, max_points: int) -> pd.DataFrame:
    if len(frame) <= max_points:
        return frame
    step = int(np.ceil(len(frame) / max_points))
    return frame.iloc[::step]


def rate_in_windows(
    times: np.ndarray,
    *,
    window: float,
    t_start: float | None = None,
    t_end: float | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    # Event rate in consecutive fixed-width windows: counts / window.
    times = np.asarray(times, dtype=float)
    times = times[np.isfinite(times)]
    if len(times) == 0 or not np.isfinite(window) or window <= 0:
        return np.array([]), np.array([])
    times = np.sort(times)
    if t_start is None:
        t_start = float(times[0])
    if t_end is None:
        t_end = float(times[-1])
    if t_end <= t_start:
        return np.array([]), np.array([])
    n_bins = int(np.floor((t_end - t_start) / window))
    if n_bins < 1:
        return np.array([]), np.array([])
    edges = t_start + np.arange(n_bins + 1, dtype=float) * window
    counts, _ = np.histogram(times, bins=edges)
    centers = 0.5 * (edges[:-1] + edges[1:])
    rate = counts.astype(float) / window
    return centers, rate



COLORS = {
    "snaps": "#4C78A8",
    "cascades": "#F58518",
    "avalanches": "#E45756",
    "JMA": "#54A24B",
    "Hauksson": "#B279A2",
}



def spatial_columns(name: str, frame: pd.DataFrame) -> tuple[pd.Series, pd.Series, str, str]:
    if name in {"JMA", "Hauksson"}:
        return frame["longitude"], frame["latitude"], "longitude", "latitude"
    return frame["x"], frame["y"], "x", "y"


# ---------------------------------------------------------------------------
# Dual time convention for simulation catalogs
# ---------------------------------------------------------------------------
# Temporal diagnostics below are shown twice:
#   1. Time domain — native occurrence times
#   2. t* = ln(t) — snaps / cascades / avalanches only (t > 0);
#      waiting times become Δln t. Quake catalogs stay on calendar time.
SIM_NAMES = ("snaps", "cascades", "avalanches")


def with_log_time(frame: pd.DataFrame) -> pd.DataFrame:
    """Map occurrence time → ln(time) for t > 0; recompute inter-event dt*."""
    out = frame.copy()
    t = out["time"].to_numpy(dtype=float)
    keep = np.isfinite(t) & (t > 0)
    out = out.loc[keep].sort_values("time", kind="mergesort").reset_index(drop=True)
    out["time"] = np.log(out["time"].to_numpy(dtype=float))
    out["dt"] = out["time"].diff()
    if "log10_dt" in out.columns:
        out["log10_dt"] = np.nan
        positive = out["dt"] > 0
        out.loc[positive, "log10_dt"] = np.log10(out.loc[positive, "dt"])
    return out


def catalogs_view(*, log_time: bool) -> dict[str, pd.DataFrame]:
    """Return CATALOGS; if log_time, remap simulation times to t* = ln(t)."""
    if not log_time:
        return CATALOGS
    return {
        name: (with_log_time(frame) if name in SIM_NAMES else frame)
        for name, frame in CATALOGS.items()
    }


CATALOGS_LN = {name: with_log_time(CATALOGS[name]) for name in SIM_NAMES}
print("CATALOGS_LN:", {k: f"n={len(v)}" for k, v in CATALOGS_LN.items()})


CATALOGS_LN: {'snaps': 'n=3242', 'cascades': 'n=892', 'avalanches': 'n=225'}


## 4. Size distributions

Empirical PDFs of the size proxy in each catalog (sim: `log10(n_swaps)` or
`log10(surface_area)` for snaps; quakes: instrumental magnitude).
Spatial-only — shown once.


In [5]:
names = list(CATALOGS)
fig = make_subplots(
    rows=1,
    cols=len(names),
    shared_yaxes=False,
    horizontal_spacing=0.04,
    subplot_titles=[
        "snaps log10(area)",
        "cascades M",
        "avalanches M",
        "JMA M",
        "Hauksson M",
    ],
)

for col, name in enumerate(names, start=1):
    mags = magnitude_series(name, CATALOGS[name]).to_numpy()
    centers, pdf = empirical_pdf(mags, bins=35, log=False)
    xlabel = "log10(surface_area)" if name == "snaps" else "magnitude"
    fig.add_trace(
        go.Scatter(
            x=centers,
            y=pdf,
            mode="lines+markers",
            marker=dict(size=5, color=COLORS[name]),
            line=dict(color=COLORS[name]),
            name=name,
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(title_text=xlabel, row=1, col=col)
    fig.update_yaxes(
        title_text="pdf" if col == 1 else "",
        type="log",
        row=1,
        col=col,
    )

fig.update_layout(
    title_text="Size / magnitude empirical PDFs",
    height=420,
    margin=dict(l=50, r=20, t=70, b=50),
)
fig.show()


## 5. Temporal structure

Inter-event timing across hierarchy levels and the two quake catalogs.
Each subsection: **time domain**, then **sims on $t^*=\ln(t)$**.


### 5.1 · $dt$ vs time

#### Time domain


In [6]:
def plot_dt_vs_time(
    catalogs: dict[str, pd.DataFrame],
    catalog_t0: dict[str, float | None],
    *,
    title: str,
    colors: dict[str, str] | None = None,
    max_points: dict[str, int | None] | None = None,
    x_title: str = "t − t₀",
    y_title: str = "dt",
    log_time_names: frozenset[str] | set[str] | None = None,
) -> go.Figure:
    colors = colors or COLORS
    log_time_names = log_time_names or frozenset()
    names = list(catalogs)
    n = len(names)
    fig = make_subplots(
        rows=n,
        cols=1,
        vertical_spacing=0.06,
        subplot_titles=[
            f"{name} [t*=ln(t)]" if name in log_time_names else name for name in names
        ],
    )
    default_max = {
        "snaps": None,
        "cascades": None,
        "avalanches": None,
        "cascades*": None,
        "avalanches*": None,
        "JMA": 60_000,
        "Hauksson": 60_000,
    }
    max_points = max_points or default_max
    for row, name in enumerate(names, start=1):
        series = dt_vs_relative_time(
            catalogs[name],
            t0=catalog_t0.get(name),
            max_points=max_points.get(name),
        )
        fig.add_trace(
            go.Scattergl(
                x=series["t"],
                y=series["dt"],
                mode="markers",
                marker=dict(size=4, opacity=0.35, color=colors.get(name, COLORS.get(name, "#333"))),
                name=name,
                showlegend=False,
            ),
            row=row,
            col=1,
        )
        fig.update_xaxes(type="log", row=row, col=1)
        fig.update_yaxes(title_text=y_title, type="log", row=row, col=1)
    fig.update_xaxes(title_text=x_title, row=n, col=1)
    x_keys = ["xaxis" if i == 1 else f"xaxis{i}" for i in range(1, n + 1)]
    y_keys = ["yaxis" if i == 1 else f"yaxis{i}" for i in range(1, n + 1)]
    fig.update_layout(
        title_text=title,
        height=280 * n,
        margin=dict(l=50, r=20, t=100, b=50),
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.0,
                xanchor="left",
                y=1.02,
                yanchor="bottom",
                pad=dict(r=8, t=0),
                showactive=True,
                buttons=[
                    dict(label="X log", method="relayout", args=[{f"{k}.type": "log" for k in x_keys}]),
                    dict(label="X linear", method="relayout", args=[{f"{k}.type": "linear" for k in x_keys}]),
                ],
            ),
            dict(
                type="buttons",
                direction="right",
                x=0.22,
                xanchor="left",
                y=1.02,
                yanchor="bottom",
                pad=dict(r=8, t=0),
                showactive=True,
                buttons=[
                    dict(label="Y log", method="relayout", args=[{f"{k}.type": "log" for k in y_keys}]),
                    dict(label="Y linear", method="relayout", args=[{f"{k}.type": "linear" for k in y_keys}]),
                ],
            ),
        ],
    )
    return fig


fig = plot_dt_vs_time(
    CATALOGS,
    CATALOG_T0,
    title="dt vs time — time domain (sim hierarchies + JMA + Hauksson)",
)
fig.show()


#### Simulations on $t^* = \ln(t)$


In [7]:
fig = plot_dt_vs_time(
    catalogs_view(log_time=True),
    {name: None for name in CATALOGS},  # relative to first event in each view
    title="dt* vs t* — simulations on t* = ln(t) (quakes unchanged)",
    x_title="t* − t*_0  (ln for sims)",
    y_title="dt*",
    log_time_names=frozenset(SIM_NAMES),
)
fig.show()


### 5.2 · Inter-event $dt$ distributions

Waiting-time PDFs:

1. **log–log** — heavy tails / power-law-like decay
2. **semi-log** (log $dt$, linear pdf) — same curves, linear vertical scale


#### Time domain


In [8]:
def plot_dt_pdfs(catalogs: dict[str, pd.DataFrame], *, title_prefix: str, dt_label: str = "dt") -> None:
    dt_curves = {}
    for name, frame in catalogs.items():
        dt = frame["dt"].to_numpy(dtype=float)
        centers, pdf = empirical_pdf(dt, bins=40, log=True)
        dt_curves[name] = (centers, pdf)

    fig = go.Figure()
    for name, (centers, pdf) in dt_curves.items():
        fig.add_trace(
            go.Scatter(
                x=centers,
                y=pdf,
                mode="lines+markers",
                marker=dict(size=5),
                line=dict(color=COLORS[name]),
                name=name,
            )
        )
    fig.update_layout(
        title=f"{title_prefix} (log–log)",
        xaxis=dict(title=dt_label, type="log"),
        yaxis=dict(title="pdf", type="log"),
        height=480,
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()

    fig = go.Figure()
    for name, (centers, pdf) in dt_curves.items():
        fig.add_trace(
            go.Scatter(
                x=centers,
                y=pdf,
                mode="lines+markers",
                marker=dict(size=5),
                line=dict(color=COLORS[name]),
                name=name,
            )
        )
    fig.update_layout(
        title=f"{title_prefix} (semi-log)",
        xaxis=dict(title=dt_label, type="log"),
        yaxis=dict(title="pdf", type="linear"),
        height=480,
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()


plot_dt_pdfs(CATALOGS, title_prefix="Empirical PDF of inter-event dt — time domain")


#### Simulations on $t^* = \ln(t)$


In [9]:
plot_dt_pdfs(
    catalogs_view(log_time=True),
    title_prefix="Empirical PDF of inter-event dt* — sims on t*=ln(t) (quakes unchanged)",
    dt_label="dt* (Δln t for sims)",
)


### 5.3 · Omori-like decays


#### Time domain


In [10]:
OMORI_SPECS = {
    "snaps": {
        "quantile": 0.50,
        "t_max_frac": 0.05,
        "fixed_m": None,
        "t_max_abs": None,
        "max_main": None,
    },
    "cascades": {
        "quantile": 0.75,
        "t_max_frac": 0.05,
        "fixed_m": None,
        "t_max_abs": None,
        "max_main": None,
    },
    "avalanches": {
        "quantile": 0.75,
        "t_max_frac": 0.05,
        "fixed_m": None,
        "t_max_abs": None,
        "max_main": None,
    },
    "JMA": {
        "quantile": None,
        "t_max_frac": None,
        "fixed_m": 4.0,
        "t_max_abs": 30 * 86400,
        "max_main": 1_000,
    },
    "Hauksson": {
        "quantile": None,
        "t_max_frac": None,
        "fixed_m": 4.0,
        "t_max_abs": 30 * 86400,
        "max_main": 1_000,
    },
}

OMORI_RNG = np.random.default_rng(0)


def plot_omori(catalogs: dict[str, pd.DataFrame], *, title: str, x_title: str) -> None:
    fig = go.Figure()
    for name, frame in catalogs.items():
        spec = OMORI_SPECS[name]
        mags = magnitude_series(name, frame).to_numpy()
        times = frame["time"].to_numpy(dtype=float)
        duration = float(times.max() - times.min()) if len(times) else 0.0
        if spec["fixed_m"] is not None:
            m_main = float(spec["fixed_m"])
            t_max = float(spec["t_max_abs"])
        else:
            finite = mags[np.isfinite(mags)]
            m_main = float(np.quantile(finite, spec["quantile"]))
            t_max = spec["t_max_frac"] * duration
        centers, rate, n_main = omori_decay(
            times,
            mags,
            m_main=m_main,
            t_max=t_max,
            max_main=spec["max_main"],
            rng=OMORI_RNG,
        )
        print(f"{name:12s} m_main={m_main:.3g}  n_main={n_main}  t_max={t_max:.3g}")
        if len(centers) == 0:
            continue
        fig.add_trace(
            go.Scatter(
                x=centers,
                y=rate,
                mode="lines+markers",
                marker=dict(size=5),
                line=dict(color=COLORS[name]),
                name=f"{name} (n={n_main})",
            )
        )
    fig.update_layout(
        title=title,
        xaxis=dict(title=x_title, type="log"),
        yaxis=dict(title="rate / mainshock", type="log"),
        height=500,
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()


plot_omori(
    CATALOGS,
    title="Omori-like stacked rate vs time since mainshock — time domain",
    x_title="t − t_main",
)


snaps        m_main=6.53  n_main=1621  t_max=2.45e+05
cascades     m_main=0.602  n_main=264  t_max=2.45e+05
avalanches   m_main=1.18  n_main=57  t_max=2.43e+05
JMA          m_main=4  n_main=1000  t_max=2.59e+06
Hauksson     m_main=4  n_main=131  t_max=2.59e+06


#### Simulations on $t^* = \ln(t)$


In [11]:
plot_omori(
    catalogs_view(log_time=True),
    title="Omori-like stacked rate — sims on t*=ln(t) (quakes unchanged)",
    x_title="t* − t*_main  (Δln t for sims)",
)


snaps        m_main=6.53  n_main=1621  t_max=0.712
cascades     m_main=0.602  n_main=264  t_max=0.67
avalanches   m_main=1.18  n_main=57  t_max=0.67
JMA          m_main=4  n_main=1000  t_max=2.59e+06
Hauksson     m_main=4  n_main=131  t_max=2.59e+06


### 5.4 · Coefficient of variation of inter-event times

$$C_v = \sigma(\Delta t)\,/\,\langle\Delta t\rangle,\qquad \Delta t = t_i - t_{i-1}$$

1. **Global** $C_v$ on the full catalog waiting times.
2. **Vs characteristic waiting time** — raise the size/magnitude threshold (mean $\Delta t$ grows with the cut).
3. **Rolling** $C_v$ of consecutive waiting times in fixed event-count windows, stamped at the window end time.

Shown twice: **time domain**, then **sims on $t^*=\ln(t)$** (waiting times $\Delta\ln t$).


#### Time domain


In [37]:
CV_WINDOWS = (50, 100, 200, 500)
CV_QUANTILES = np.linspace(0.0, 0.9, 10)
CV_WINDOW_COLORS = {
    50: "#9ecae1",
    100: "#6baed6",
    200: "#3182bd",
    500: "#08519c",
}


def compute_cv_bundle(catalogs: dict[str, pd.DataFrame]):
    cv_global_rows = []
    cv_scale_curves: dict[str, pd.DataFrame] = {}
    cv_ts: dict[str, dict[int, tuple[np.ndarray, np.ndarray]]] = {}
    for name, frame in catalogs.items():
        times = frame["time"].to_numpy(dtype=float)
        dt = frame["dt"].to_numpy(dtype=float)
        cv_all = interevent_cv(dt)
        finite_dt = dt[np.isfinite(dt) & (dt > 0)]
        cv_global_rows.append(
            {
                "catalog": name,
                "n_dt": int(len(finite_dt)),
                "mean_dt": float(np.mean(finite_dt)) if len(finite_dt) else np.nan,
                "cv": cv_all,
            }
        )
        sizes = magnitude_series(name, frame).to_numpy(dtype=float)
        finite_sizes = sizes[np.isfinite(sizes)]
        if len(finite_sizes) >= 50:
            thresholds = np.unique(np.quantile(finite_sizes, CV_QUANTILES))
            cv_scale_curves[name] = cv_vs_size_threshold(
                times, sizes, thresholds, min_events=50
            )
        per_window: dict[int, tuple[np.ndarray, np.ndarray]] = {}
        for window in CV_WINDOWS:
            per_window[window] = cv_time_series(times, dt, window=window)
        cv_ts[name] = per_window
        n_by_w = ", ".join(
            f"W={w}:{len(per_window[w][0])}" for w in CV_WINDOWS if len(per_window[w][0])
        )
        print(
            f"{name:12s}  C_v={cv_all:.3f}  "
            f"mean_dt={np.mean(finite_dt):.3g}  "
            f"scale_pts={len(cv_scale_curves.get(name, []))}  "
            f"{n_by_w}"
        )
    return pd.DataFrame(cv_global_rows), cv_scale_curves, cv_ts


def plot_cv_global_and_scale(cv_global, cv_scale_curves, *, title_suffix: str) -> None:
    display(cv_global)
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=cv_global["catalog"],
            y=cv_global["cv"],
            marker_color=[COLORS[name] for name in cv_global["catalog"]],
            name="C_v",
        )
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", annotation_text="Poisson C_v=1")
    fig.update_layout(
        title=f"Global C_v of inter-event waiting times — {title_suffix}",
        yaxis_title="C_v",
        height=420,
        margin=dict(l=60, r=20, t=50, b=50),
        showlegend=False,
    )
    fig.show()

    fig = go.Figure()
    for name, scale in cv_scale_curves.items():
        if scale.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=scale["mean_dt"],
                y=scale["cv"],
                mode="lines+markers",
                marker=dict(size=7),
                line=dict(color=COLORS[name]),
                name=name,
                customdata=np.column_stack([scale["threshold"], scale["n_events"]]),
                hovertemplate=(
                    "%{fullData.name}<br>⟨Δt⟩=%{x:.3g}<br>C_v=%{y:.3f}"
                    "<br>threshold=%{customdata[0]:.3g}<br>n=%{customdata[1]:.0f}"
                    "<extra></extra>"
                ),
            )
        )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"C_v vs mean waiting time — {title_suffix}",
        xaxis=dict(title="⟨waiting time⟩", type="log"),
        yaxis=dict(title="C_v", type="linear"),
        height=500,
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()

    fig = go.Figure()
    for name, scale in cv_scale_curves.items():
        if scale.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=scale["threshold"],
                y=scale["cv"],
                mode="lines+markers",
                marker=dict(size=7),
                line=dict(color=COLORS[name]),
                name=name,
                customdata=np.column_stack([scale["mean_dt"], scale["n_events"]]),
                hovertemplate=(
                    "%{fullData.name}<br>threshold=%{x:.3g}<br>C_v=%{y:.3f}"
                    "<br>⟨Δt⟩=%{customdata[0]:.3g}<br>n=%{customdata[1]:.0f}"
                    "<extra></extra>"
                ),
            )
        )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray")
    fig.update_layout(
        title=f"C_v vs size / magnitude threshold — {title_suffix}",
        xaxis=dict(title="size / magnitude threshold", type="linear"),
        yaxis=dict(title="C_v", type="linear"),
        height=500,
        margin=dict(l=60, r=20, t=50, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()


def plot_cv_rolling(
    cv_ts,
    catalog_names,
    *,
    title: str,
    log_time_names: frozenset[str] | set[str] | None = None,
) -> None:
    log_time_names = log_time_names or frozenset()
    fig = make_subplots(
        rows=len(catalog_names),
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.06,
        subplot_titles=[
            f"{name} [t*=ln(t)]" if name in log_time_names else name
            for name in catalog_names
        ],
    )
    for row, name in enumerate(catalog_names, start=1):
        use_datetime = name in {"JMA", "Hauksson"} and name not in log_time_names
        for window in CV_WINDOWS:
            t_cv, cv_vals = cv_ts[name][window]
            if len(t_cv) == 0:
                continue
            step = max(1, len(t_cv) // 5000)
            x = t_cv[::step]
            if use_datetime:
                x = pd.to_datetime(x, unit="s")
            fig.add_trace(
                go.Scattergl(
                    x=x,
                    y=cv_vals[::step],
                    mode="lines",
                    line=dict(color=CV_WINDOW_COLORS[window], width=1.2),
                    name=f"W={window}",
                    legendgroup=f"W={window}-{title}",
                    showlegend=(row == 1),
                ),
                row=row,
                col=1,
            )
        fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=row, col=1)
        fig.update_yaxes(title_text="C_v", row=row, col=1)
        if name in log_time_names:
            xlab = "t* = ln(t)"
        elif use_datetime:
            xlab = "time"
        else:
            xlab = "time (sim)"
        fig.update_xaxes(title_text=xlab, row=row, col=1)
    fig.update_layout(
        title_text=title,
        height=220 * len(catalog_names),
        margin=dict(l=60, r=20, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig.show()


print("=== C_v — time domain ===")
cv_global, cv_scale_curves, cv_ts = compute_cv_bundle(CATALOGS)
plot_cv_global_and_scale(cv_global, cv_scale_curves, title_suffix="time domain")
plot_cv_rolling(
    cv_ts,
    list(CATALOGS),
    title="Rolling C_v(t) — time domain",
)


=== C_v — time domain ===
snaps         C_v=14.407  mean_dt=1.51e+03  scale_pts=10  W=50:3189, W=100:3139, W=200:3039, W=500:2739
cascades      C_v=7.513  mean_dt=5.49e+03  scale_pts=5  W=50:842, W=100:792, W=200:692, W=500:392
avalanches    C_v=3.777  mean_dt=2.17e+04  scale_pts=6  W=50:175, W=100:125, W=200:25
JMA           C_v=1.464  mean_dt=1.02e+03  scale_pts=10  W=50:154157, W=100:154107, W=200:154007, W=500:153707
Hauksson      C_v=1.701  mean_dt=2.59e+03  scale_pts=10  W=50:60849, W=100:60799, W=200:60699, W=500:60399


,catalog,n_dt,mean_dt,cv
0,snaps,3238,1510.784940,14.407299
1,cascades,891,5490.359594,7.512684
2,avalanches,224,21655.569850,3.776969
3,JMA,154206,1023.059557,1.464005
4,Hauksson,60898,2590.640432,1.700703


#### Simulations on $t^* = \ln(t)$


In [36]:
print("=== C_v — simulations on t* = ln(t) ===")
cv_global_ln, cv_scale_ln, cv_ts_ln = compute_cv_bundle(catalogs_view(log_time=True))
plot_cv_global_and_scale(
    cv_global_ln,
    cv_scale_ln,
    title_suffix="sims on t*=ln(t) (quakes unchanged)",
)
plot_cv_rolling(
    cv_ts_ln,
    list(CATALOGS),
    title="Rolling C_v — sims on t*=ln(t) (quakes unchanged)",
    log_time_names=frozenset(SIM_NAMES),
)


=== C_v — simulations on t* = ln(t) ===
snaps         C_v=4.704  mean_dt=0.0044  scale_pts=10  W=50:3189, W=100:3139, W=200:3039, W=500:2739
cascades      C_v=3.065  mean_dt=0.015  scale_pts=5  W=50:842, W=100:792, W=200:692, W=500:392
avalanches    C_v=1.345  mean_dt=0.0598  scale_pts=6  W=50:175, W=100:125, W=200:25
JMA           C_v=1.464  mean_dt=1.02e+03  scale_pts=10  W=50:154157, W=100:154107, W=200:154007, W=500:153707
Hauksson      C_v=1.701  mean_dt=2.59e+03  scale_pts=10  W=50:60849, W=100:60799, W=200:60699, W=500:60399


,catalog,n_dt,mean_dt,cv
0,snaps,3238,0.004399,4.704164
1,cascades,891,0.015039,3.064975
2,avalanches,224,0.059782,1.344628
3,JMA,154206,1023.059557,1.464005
4,Hauksson,60898,2590.640432,1.700703


### 5.5 · Rate per time window

Event rate in consecutive fixed-width windows: $\mathrm{rate} = N_{\mathrm{window}} / \Delta T$.

| Catalog | Windows |
|---|---|
| JMA / Hauksson | 1 day, 1 week, 1 month |
| snaps / cascades / avalanches | same fractions of catalog duration as day/week/month are of a 5-year span |

Second pass: sims on $u=\ln(t)$ with window widths as the **same duration fractions of the $u$-span**. Quake panels stay on calendar time.


#### Time domain


In [38]:
SEC_DAY = 86400.0
SEC_WEEK = 7.0 * SEC_DAY
SEC_MONTH = 30.4375 * SEC_DAY
RATE_QUAKE_WINDOWS = {
    "day": SEC_DAY,
    "week": SEC_WEEK,
    "month": SEC_MONTH,
}
RATE_REF_SPAN = 5.0 * 365.25 * SEC_DAY
RATE_FRACTIONS = {name: width / RATE_REF_SPAN for name, width in RATE_QUAKE_WINDOWS.items()}
RATE_WINDOW_COLORS = {
    "day": "#9ecae1",
    "week": "#3182bd",
    "month": "#08519c",
}


def rate_windows_for(name: str, times: np.ndarray, *, log_time: bool) -> dict[str, float]:
    times = np.asarray(times, dtype=float)
    times = times[np.isfinite(times)]
    if log_time:
        times = times[times > 0]
        if len(times) < 2:
            return {}
        span = float(np.log(times.max()) - np.log(times.min()))
        return {label: frac * span for label, frac in RATE_FRACTIONS.items()}
    if name in {"JMA", "Hauksson"}:
        return dict(RATE_QUAKE_WINDOWS)
    if len(times) < 2:
        return {}
    span = float(times.max() - times.min())
    return {label: frac * span for label, frac in RATE_FRACTIONS.items()}


def rate_series_for(
    name: str,
    frame: pd.DataFrame,
    *,
    log_time: bool,
) -> dict[str, tuple[np.ndarray, np.ndarray, float]]:
    times = frame["time"].to_numpy(dtype=float)
    if log_time:
        times = times[np.isfinite(times) & (times > 0)]
        coord = np.log(np.sort(times))
    else:
        times = times[np.isfinite(times)]
        coord = np.sort(times)
    windows = rate_windows_for(name, frame["time"].to_numpy(dtype=float), log_time=log_time)
    out: dict[str, tuple[np.ndarray, np.ndarray, float]] = {}
    for label, width in windows.items():
        centers, rate = rate_in_windows(coord, window=width)
        out[label] = (centers, rate, width)
    return out


def plot_rate_subplots(
    series_by_catalog: dict[str, dict[str, tuple[np.ndarray, np.ndarray, float]]],
    *,
    title: str,
    log_time_names: frozenset[str] | set[str],
) -> go.Figure:
    names = list(series_by_catalog)
    fig = make_subplots(
        rows=len(names),
        cols=1,
        shared_xaxes=False,
        vertical_spacing=0.06,
        subplot_titles=[
            f"{name} [t*=ln(t)]" if name in log_time_names else name for name in names
        ],
    )
    for row, name in enumerate(names, start=1):
        use_datetime = name in {"JMA", "Hauksson"} and name not in log_time_names
        for label, (centers, rate, width) in series_by_catalog[name].items():
            if len(centers) == 0:
                continue
            step = max(1, len(centers) // 5000)
            x = centers[::step]
            if use_datetime:
                x = pd.to_datetime(x, unit="s")
            fig.add_trace(
                go.Scattergl(
                    x=x,
                    y=rate[::step],
                    mode="lines",
                    line=dict(color=RATE_WINDOW_COLORS[label], width=1.2),
                    name=label,
                    legendgroup=f"rate-{title}-{label}",
                    showlegend=(row == 1),
                    hovertemplate=(
                        f"{name} · {label}<br>x=%{{x}}<br>rate=%{{y:.3g}}"
                        f"<br>Δ={width:.3g}<extra></extra>"
                    ),
                ),
                row=row,
                col=1,
            )
        ylab = "rate / dln(t)" if name in log_time_names else "rate / time"
        fig.update_yaxes(title_text=ylab, type="log", row=row, col=1)
        if name in log_time_names:
            xlab = "t* = ln(t)"
        elif use_datetime:
            xlab = "time"
        else:
            xlab = "time (sim)"
        fig.update_xaxes(title_text=xlab, row=row, col=1)
    fig.update_layout(
        title_text=title,
        height=220 * len(names),
        margin=dict(l=60, r=20, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    return fig


print("=== Rate — time domain ===")
rate_linear: dict[str, dict[str, tuple[np.ndarray, np.ndarray, float]]] = {}
for name, frame in CATALOGS.items():
    rate_linear[name] = rate_series_for(name, frame, log_time=False)
    widths = {k: v[2] for k, v in rate_linear[name].items()}
    print(
        f"{name:12s}  linear windows  "
        + ", ".join(f"{k}={w:.4g}" for k, w in widths.items())
    )

fig = plot_rate_subplots(
    rate_linear,
    title="Event rate — time domain (sim: duration-fraction day/week/month)",
    log_time_names=frozenset(),
)
fig.show()


=== Rate — time domain ===
snaps         linear windows  day=2679, week=1.875e+04, month=8.153e+04
cascades      linear windows  day=2679, week=1.875e+04, month=8.153e+04
avalanches    linear windows  day=2656, week=1.859e+04, month=8.085e+04
JMA           linear windows  day=8.64e+04, week=6.048e+05, month=2.63e+06
Hauksson      linear windows  day=8.64e+04, week=6.048e+05, month=2.63e+06


#### Simulations on $t^* = \ln(t)$


In [39]:
print("=== Rate — simulations on t* = ln(t) ===")
rate_logish: dict[str, dict[str, tuple[np.ndarray, np.ndarray, float]]] = {}
for name, frame in CATALOGS.items():
    if name in SIM_NAMES:
        rate_logish[name] = rate_series_for(name, frame, log_time=True)
        widths = {k: v[2] for k, v in rate_logish[name].items()}
        print(
            f"{name:12s}  ln(time) windows  "
            + ", ".join(f"{k}={w:.4g}" for k, w in widths.items())
        )
    else:
        rate_logish[name] = rate_linear[name]

fig = plot_rate_subplots(
    rate_logish,
    title="Event rate — sims on t*=ln(t) (quakes unchanged)",
    log_time_names=frozenset(SIM_NAMES),
)
fig.show()


=== Rate — simulations on t* = ln(t) ===
snaps         ln(time) windows  day=0.0078, week=0.0546, month=0.2374
cascades      ln(time) windows  day=0.007337, week=0.05136, month=0.2233
avalanches    ln(time) windows  day=0.007333, week=0.05133, month=0.2232


## 6. Rudyak et al. age-binned correlations (Fig. 2)

Recreate the top panel of Rudyak, Shohat & Lahini (2025) Fig. 2 for **snaps**:
waiting-time PDFs conditioned on system age, using half-decade log windows

$$t \in [10^{k},\, 10^{k+0.5}),\qquad k = 2.0, 2.5, \ldots$$

plus the joint $P(\Delta t, \Delta r)$ and marginal $P(\Delta r)$.

- $\Delta t_i = t_i - t_{i-1}$ assigned to the age window of event $i$
- $\Delta t^\ast \approx 10$ (cascade `max_time_gap` / paper crossover)
- $R_c$ marked where $P(\Delta r)$ flattens (empirical)

Defines `AGE_WINDOWS` / `active_age_windows` / `AGE_COLORS` reused in §§8–10.


In [16]:
# Rudyak et al. (2025) Fig. 2 — age-binned P(Δt) + joint P(Δt, Δr) for snaps.

DELTA_T_STAR = float(settings.cascades.max_time_gap)  # ≈ 10 in trial32 / paper
AGE_LOG_STEP = 0.5
AGE_LOG_START = 2.0
AGE_LOG_STOP = 7.0  # exclusive upper edge of last half-decade
AGE_MIN_EVENTS = 40
AGE_CATALOG = "snaps"

# Half-decade edges in log10(t), matching the Fig. 2 top legend style.
_age_edges = np.arange(AGE_LOG_START, AGE_LOG_STOP + 1e-9, AGE_LOG_STEP)
AGE_WINDOWS: list[tuple[float, float, str]] = []
def _fmt_log10(edge: float) -> str:
    # Compact decade labels for plotly (plain text, not LaTeX).
    if float(edge).is_integer():
        return f"10^{int(edge)}"
    return f"10^{edge:g}"


for lo, hi in zip(_age_edges[:-1], _age_edges[1:]):
    label = f"[{_fmt_log10(lo)}, {_fmt_log10(hi)})"
    AGE_WINDOWS.append((10.0 ** lo, 10.0 ** hi, label))


def age_window_colors(n: int) -> list[str]:
    """Warm → cool sequence similar to Rudyak Fig. 2."""
    # Plotly / matplotlib-independent discrete palette.
    base = [
        "#F0E442",
        "#E69F00",
        "#D55E00",
        "#CC79A7",
        "#C44E52",
        "#8172B3",
        "#4C72B0",
        "#2C4E80",
        "#1B1B3A",
        "#0B1320",
    ]
    if n <= len(base):
        return base[:n]
    # Repeat-extend if more windows than colors.
    return [base[i % len(base)] for i in range(n)]


def consecutive_pairs(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per consecutive event pair with Δt, Δr, and child age t."""
    need = {"time", "x", "y"}
    missing = need - set(frame.columns)
    if missing:
        raise KeyError(f"consecutive_pairs needs columns {sorted(need)}; missing {missing}")
    ordered = frame.sort_values("time", kind="mergesort").reset_index(drop=True)
    t = ordered["time"].to_numpy(dtype=float)
    x = ordered["x"].to_numpy(dtype=float)
    y = ordered["y"].to_numpy(dtype=float)
    if len(t) < 2:
        return pd.DataFrame(columns=["t", "dt", "dr"])
    dt = np.diff(t)
    dr = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
    child_t = t[1:]
    ok = np.isfinite(dt) & np.isfinite(dr) & (dt > 0) & (dr >= 0) & np.isfinite(child_t)
    return pd.DataFrame({"t": child_t[ok], "dt": dt[ok], "dr": dr[ok]})


def slice_by_age(frame: pd.DataFrame, t_lo: float, t_hi: float) -> pd.DataFrame:
    t = frame["time"].to_numpy(dtype=float)
    return frame.loc[np.isfinite(t) & (t >= t_lo) & (t < t_hi)].copy()


def active_age_windows(
    frame: pd.DataFrame,
    *,
    min_events: int = AGE_MIN_EVENTS,
) -> list[tuple[float, float, str, pd.DataFrame]]:
    out = []
    for t_lo, t_hi, label in AGE_WINDOWS:
        sub = slice_by_age(frame, t_lo, t_hi)
        if len(sub) >= min_events:
            out.append((t_lo, t_hi, label, sub))
    return out


snap_pairs = consecutive_pairs(CATALOGS[AGE_CATALOG])
active_windows = active_age_windows(CATALOGS[AGE_CATALOG])
AGE_COLORS = {
    label: color
    for color, (*_, label, _) in zip(age_window_colors(len(active_windows)), active_windows)
}

print(f"Δt* = {DELTA_T_STAR:g}  (cascade max_time_gap)")
print(f"{AGE_CATALOG} consecutive pairs: {len(snap_pairs):,}")
for t_lo, t_hi, label, sub in active_windows:
    print(f"  t ∈ {label}: n={len(sub):,}")


def estimate_rc(dr: np.ndarray, *, bins: int = 30) -> float:
    """Rough Rc: first scale where log-binned P(Δr) slope flattens."""
    centers, pdf = empirical_pdf(dr, bins=bins, log=True)
    if len(centers) < 4:
        return float("nan")
    log_c = np.log10(centers)
    log_p = np.log10(pdf)
    # local slope over 3-point windows; Rc ~ first center after slope > -0.3
    slopes = np.diff(log_p) / np.diff(log_c)
    flat = np.flatnonzero(slopes > -0.3)
    if len(flat) == 0:
        return float(np.nanmedian(dr))
    return float(centers[min(int(flat[0]) + 1, len(centers) - 1)])


RC_EST = estimate_rc(snap_pairs["dr"].to_numpy(dtype=float))
print(f"estimated Rc ≈ {RC_EST:.3g}")

# --- Fig. 2 layout: top P(Δt | age), main joint, right P(Δr) ---
fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=True,
    shared_yaxes=False,
    column_widths=[0.78, 0.22],
    row_heights=[0.38, 0.62],
    horizontal_spacing=0.04,
    vertical_spacing=0.06,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
    ],
    subplot_titles=("P(Δt) by age window", "", "P(Δt, Δr)", "P(Δr)"),
)

# Top: age-conditioned waiting-time PDFs
for t_lo, t_hi, label, _sub in active_windows:
    mask = (snap_pairs["t"] >= t_lo) & (snap_pairs["t"] < t_hi)
    centers, pdf = empirical_pdf(snap_pairs.loc[mask, "dt"].to_numpy(dtype=float), bins=35, log=True)
    if len(centers) == 0:
        continue
    fig.add_trace(
        go.Scatter(
            x=centers,
            y=pdf,
            mode="lines",
            line=dict(color=AGE_COLORS[label], width=2),
            name=f"t ∈ {label}",
            legendgroup=label,
        ),
        row=1,
        col=1,
    )

# Δt* guide on top panel
fig.add_vline(
    x=DELTA_T_STAR,
    line=dict(color="black", width=1.5, dash="dash"),
    row=1,
    col=1,
)

# Main joint density on log–log bins
dt_all = snap_pairs["dt"].to_numpy(dtype=float)
dr_all = snap_pairs["dr"].to_numpy(dtype=float)
ok = (dt_all > 0) & (dr_all > 0) & np.isfinite(dt_all) & np.isfinite(dr_all)
dt_all = dt_all[ok]
dr_all = dr_all[ok]
n_dt_bins, n_dr_bins = 48, 36
dt_edges = np.logspace(np.log10(dt_all.min()), np.log10(dt_all.max()), n_dt_bins + 1)
dr_edges = np.logspace(np.log10(max(dr_all.min(), 1e-12)), np.log10(dr_all.max()), n_dr_bins + 1)
counts, _, _ = np.histogram2d(np.log10(dt_all), np.log10(dr_all), bins=[np.log10(dt_edges), np.log10(dr_edges)])
# Convert counts → density in linear (Δt, Δr) measure
dt_w = np.diff(dt_edges)
dr_w = np.diff(dr_edges)
area = np.outer(dt_w, dr_w)
dens = counts / (counts.sum() * area)
with np.errstate(divide="ignore", invalid="ignore"):
    dens_plot = np.where(dens > 0, np.log10(dens), np.nan)
dt_centers = np.sqrt(dt_edges[:-1] * dt_edges[1:])
dr_centers = np.sqrt(dr_edges[:-1] * dr_edges[1:])

fig.add_trace(
    go.Heatmap(
        x=dt_centers,
        y=dr_centers,
        z=dens_plot.T,
        colorscale="Hot_r",
        colorbar=dict(title="log₁₀ PDF", len=0.45, y=0.22),
        hovertemplate="Δt=%{x:.3g}<br>Δr=%{y:.3g}<br>log10 PDF=%{z:.2f}<extra></extra>",
        name="joint",
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Regime guides on joint panel
fig.add_vline(x=DELTA_T_STAR, line=dict(color="black", width=1.2, dash="dash"), row=2, col=1)
if np.isfinite(RC_EST) and RC_EST > 0:
    fig.add_hline(y=RC_EST, line=dict(color="black", width=1.2, dash="dash"), row=2, col=1)

# Regime labels as figure annotations (paper: mechanical / thermal / uncorrelated)
fig.add_annotation(
    text=f"Δt*={DELTA_T_STAR:g}" + (f"   Rc≈{RC_EST:.3g}" if np.isfinite(RC_EST) else ""),
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.01,
    showarrow=False,
    font=dict(size=12),
    xanchor="left",
)

# Right: marginal P(Δr)
dr_c, dr_pdf = empirical_pdf(dr_all, bins=35, log=True)
fig.add_trace(
    go.Scatter(
        x=dr_pdf,
        y=dr_c,
        mode="lines",
        line=dict(color="#4C72B0", width=2),
        name="P(Δr)",
        showlegend=False,
    ),
    row=2,
    col=2,
)
if np.isfinite(RC_EST) and RC_EST > 0:
    fig.add_hline(y=RC_EST, line=dict(color="black", width=1.2, dash="dash"), row=2, col=2)

fig.update_xaxes(type="log", title_text="Δt", row=1, col=1)
fig.update_yaxes(type="log", title_text="P(Δt)", row=1, col=1)
fig.update_xaxes(type="log", title_text="Δt", row=2, col=1)
fig.update_yaxes(type="log", title_text="Δr", row=2, col=1)
fig.update_xaxes(type="log", title_text="P(Δr)", row=2, col=2)
fig.update_yaxes(type="log", title_text="Δr", row=2, col=2)
# Hide unused top-right panel axes
fig.update_xaxes(visible=False, row=1, col=2)
fig.update_yaxes(visible=False, row=1, col=2)

fig.update_layout(
    title=f"Rudyak et al. Fig. 2 — snaps: age-binned P(Δt) and P(Δt, Δr)  (Δt*={DELTA_T_STAR:g})",
    height=820,
    width=980,
    legend=dict(orientation="v", yanchor="top", y=0.98, x=1.02),
    margin=dict(l=70, r=160, t=60, b=50),
    template="plotly_white",
)
fig.show()


Δt* = 10  (cascade max_time_gap)
snaps consecutive pairs: 3,238
  t ∈ [10^2, 10^2.5): n=94
  t ∈ [10^2.5, 10^3): n=217
  t ∈ [10^3, 10^3.5): n=573
  t ∈ [10^3.5, 10^4): n=1,099
  t ∈ [10^4, 10^4.5): n=391
  t ∈ [10^4.5, 10^5): n=187
  t ∈ [10^5, 10^5.5): n=68
  t ∈ [10^6, 10^6.5): n=417
  t ∈ [10^6.5, 10^7): n=55
estimated Rc ≈ 4.67


## 7. Spatiotemporal views

Catalogs in a subplot grid. Time-dependent panels are shown twice (**time domain**, then **sims on $t^*=\ln(t)$**). The map view is spatial-only and appears once.

1. latitude / `y` vs time
2. longitude / `x` vs time
3. magnitude / size vs time — **snaps: not available**
4. latitude / `y` vs longitude / `x` (map view)


### 7.1 · Space vs time

#### Time domain


In [17]:
def relative_time(frame: pd.DataFrame, t0: float | None = None) -> pd.Series:
    if t0 is None:
        t0 = float(frame["time"].iloc[0])
    return frame["time"] - float(t0)


PANEL_MAX = {
    "snaps": 8_000,
    "cascades": 8_000,
    "avalanches": 8_000,
    "JMA": 40_000,
    "Hauksson": 40_000,
}


def build_plot_frames(
    catalogs: dict[str, pd.DataFrame],
    catalog_t0: dict[str, float | None],
) -> dict[str, pd.DataFrame]:
    plot_frames: dict[str, pd.DataFrame] = {}
    for name, frame in catalogs.items():
        plot_frame = downsample(frame, PANEL_MAX[name]).copy()
        plot_frame["t_rel"] = relative_time(plot_frame, t0=catalog_t0.get(name))
        lon_like, lat_like, lon_label, lat_label = spatial_columns(name, plot_frame)
        plot_frame["lon_like"] = lon_like.to_numpy()
        plot_frame["lat_like"] = lat_like.to_numpy()
        plot_frame["size"] = magnitude_series(name, plot_frame).to_numpy()
        plot_frame.attrs["lon_label"] = lon_label
        plot_frame.attrs["lat_label"] = lat_label
        plot_frames[name] = plot_frame
    return plot_frames


def show_spatiotemporal_panels(
    plot_frames: dict[str, pd.DataFrame],
    panel_specs,
    *,
    title_suffix: str,
) -> None:
    names = list(plot_frames)
    n = len(names)
    N_ROWS, N_COLS = 2, 3
    for title, x_key, y_key, x_title, y_title, square in panel_specs:
        fig = make_subplots(
            rows=N_ROWS,
            cols=N_COLS,
            subplot_titles=[
                (
                    "snaps — not available"
                    if (y_key == "size" and name == "snaps")
                    else f"{name} (n={len(plot_frames[name]):,})"
                )
                for name in names
            ]
            + [""] * (N_ROWS * N_COLS - n),
            horizontal_spacing=0.08,
            vertical_spacing=0.12,
        )
        size_panel = y_key == "size"
        for index, name in enumerate(names):
            row = index // N_COLS + 1
            col = index % N_COLS + 1
            pf = plot_frames[name]
            axis_id = index + 1
            xaxis_name = "x" if axis_id == 1 else f"x{axis_id}"
            if size_panel and name == "snaps":
                fig.add_annotation(
                    text="not available",
                    x=0.5,
                    y=0.5,
                    xref="x domain",
                    yref="y domain",
                    showarrow=False,
                    font=dict(size=14, color="#666666"),
                    row=row,
                    col=col,
                )
            else:
                fig.add_trace(
                    go.Scattergl(
                        x=pf[x_key],
                        y=pf[y_key],
                        mode="markers",
                        marker=dict(size=3, opacity=0.35, color=COLORS[name]),
                        name=name,
                        showlegend=False,
                    ),
                    row=row,
                    col=col,
                )
            fig.update_xaxes(title_text=x_title, row=row, col=col)
            fig.update_yaxes(
                title_text=y_title,
                scaleanchor=xaxis_name if square else None,
                scaleratio=1 if square else None,
                row=row,
                col=col,
            )
        fig.update_layout(
            title_text=f"{title} — {title_suffix}",
            height=720,
            margin=dict(l=60, r=20, t=70, b=50),
        )
        fig.show()


TIME_PANEL_SPECS = (
    ("lat / y vs time", "t_rel", "lat_like", "t − t₀", "lat / y", False),
    ("lon / x vs time", "t_rel", "lon_like", "t − t₀", "lon / x", False),
    ("size / magnitude vs time", "t_rel", "size", "t − t₀", "size / magnitude", False),
)
MAP_PANEL_SPECS = (
    ("lat / y vs lon / x", "lon_like", "lat_like", "lon / x", "lat / y", True),
)

plot_frames = build_plot_frames(CATALOGS, CATALOG_T0)
show_spatiotemporal_panels(
    plot_frames,
    TIME_PANEL_SPECS,
    title_suffix="time domain",
)


#### Simulations on $t^* = \ln(t)$


In [18]:
plot_frames_ln = build_plot_frames(
    catalogs_view(log_time=True),
    {name: None for name in CATALOGS},
)
# Relabel x for sims on ln time
TIME_PANEL_SPECS_LN = (
    ("lat / y vs t*", "t_rel", "lat_like", "t* − t*_0 (ln for sims)", "lat / y", False),
    ("lon / x vs t*", "t_rel", "lon_like", "t* − t*_0 (ln for sims)", "lon / x", False),
    ("size / magnitude vs t*", "t_rel", "size", "t* − t*_0 (ln for sims)", "size / magnitude", False),
)
show_spatiotemporal_panels(
    plot_frames_ln,
    TIME_PANEL_SPECS_LN,
    title_suffix="sims on t*=ln(t) (quakes unchanged)",
)


### 7.2 · Map view (time-independent)


In [19]:
# Map view is purely spatial — shown once (no time remapping).
show_spatiotemporal_panels(
    plot_frames,
    MAP_PANEL_SPECS,
    title_suffix="map view (time-independent)",
)


## 8. Spatial fractal dimension (box-counting)

Rasterize each catalog's 2D locations onto a square occupancy grid, then
estimate the Minkowski–Bouligand (box-counting) dimension with
`skimage.measure.block_reduce`:

$$N(\varepsilon) \propto \varepsilon^{-D}$$

so $D = -$slope of $\log N$ vs $\log \varepsilon$.

- sim catalogs → sheet coordinates `(x, y)`
- JMA / Hauksson → `(longitude, latitude)`
- **snaps by Rudyak age window** (§6) → extra scaling curves + occupancy subplots

Absolute map projection differs; compare $D$ and the scaling curves, not
absolute box sizes.


In [20]:
def xy_coords(name: str, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    lon_like, lat_like, _, _ = spatial_columns(name, frame)
    x = np.asarray(lon_like, dtype=float)
    y = np.asarray(lat_like, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]


def occupancy_grid(
    x: np.ndarray,
    y: np.ndarray,
    *,
    n_bins: int = 512,
    x_range: tuple[float, float] | None = None,
    y_range: tuple[float, float] | None = None,
) -> np.ndarray:
    """Binary occupancy image on a square bounding box.

    Optional ``x_range`` / ``y_range`` fix the box (useful for age-window
    subplots that should share the full-catalog spatial extent).
    """

    if x_range is None:
        xmin, xmax = float(x.min()), float(x.max())
    else:
        xmin, xmax = map(float, x_range)
    if y_range is None:
        ymin, ymax = float(y.min()), float(y.max())
    else:
        ymin, ymax = map(float, y_range)
    span = max(xmax - xmin, ymax - ymin)
    if span <= 0:
        span = 1.0
    x0 = 0.5 * (xmin + xmax) - 0.5 * span
    y0 = 0.5 * (ymin + ymax) - 0.5 * span
    hist, _, _ = np.histogram2d(
        y,
        x,
        bins=n_bins,
        range=[[y0, y0 + span], [x0, x0 + span]],
    )
    return hist > 0


def box_counting_dimension(
    binary: np.ndarray,
    *,
    min_box: int = 2,
) -> tuple[float, np.ndarray, np.ndarray]:
    """Minkowski–Bouligand dimension via skimage block_reduce.

    Returns ``(D, box_sizes, occupied_counts)``.
    """

    binary = np.asarray(binary, dtype=bool)
    if binary.ndim != 2:
        raise ValueError("binary occupancy must be 2-D")
    side = max(binary.shape)
    n2 = 1 << int(np.ceil(np.log2(max(side, 2))))
    padded = np.zeros((n2, n2), dtype=bool)
    padded[: binary.shape[0], : binary.shape[1]] = binary

    sizes: list[int] = []
    counts: list[int] = []
    size = n2 // 2  # skip the trivial whole-image box
    while size >= min_box:
        reduced = measure.block_reduce(padded, block_size=(size, size), func=np.any)
        n_occ = int(np.count_nonzero(reduced))
        if n_occ > 0:
            sizes.append(size)
            counts.append(n_occ)
        size //= 2

    sizes_arr = np.asarray(sizes, dtype=float)
    counts_arr = np.asarray(counts, dtype=float)
    if len(sizes_arr) < 2:
        return float("nan"), sizes_arr, counts_arr
    slope = float(np.polyfit(np.log(sizes_arr), np.log(counts_arr), 1)[0])
    return -slope, sizes_arr, counts_arr


FRACTAL_BINS = 512
fractal_rows = []
scaling = {}

for name, frame in CATALOGS.items():
    x, y = xy_coords(name, frame)
    occ = occupancy_grid(x, y, n_bins=FRACTAL_BINS)
    dim, sizes, counts = box_counting_dimension(occ)
    fractal_rows.append(
        {
            "catalog": name,
            "n_events": len(x),
            "occupied_pixels": int(occ.sum()),
            "grid": FRACTAL_BINS,
            "D_box": dim,
        }
    )
    scaling[name] = (sizes, counts, dim)
    print(
        f"{name:12s}  D_box={dim:6.3f}  "
        f"occupied={int(occ.sum()):6d}/{FRACTAL_BINS**2}  n={len(x):,}"
    )

# Per Rudyak age window (snaps), sharing the full-catalog bounding box.
snap_x_all, snap_y_all = xy_coords("snaps", CATALOGS["snaps"])
SNAP_XY_RANGE = (
    (float(snap_x_all.min()), float(snap_x_all.max())),
    (float(snap_y_all.min()), float(snap_y_all.max())),
)
age_scaling: dict[str, tuple[np.ndarray, np.ndarray, float]] = {}
age_occupancy: dict[str, np.ndarray] = {}
age_fractal_rows = []

for t_lo, t_hi, label, sub in active_age_windows(CATALOGS["snaps"]):
    x, y = xy_coords("snaps", sub)
    if len(x) < 8:
        continue
    occ = occupancy_grid(
        x,
        y,
        n_bins=FRACTAL_BINS,
        x_range=SNAP_XY_RANGE[0],
        y_range=SNAP_XY_RANGE[1],
    )
    dim, sizes, counts = box_counting_dimension(occ)
    key = f"snaps {label}"
    age_scaling[key] = (sizes, counts, dim)
    age_occupancy[key] = occ
    age_fractal_rows.append(
        {
            "catalog": key,
            "t_lo": t_lo,
            "t_hi": t_hi,
            "n_events": len(x),
            "occupied_pixels": int(occ.sum()),
            "grid": FRACTAL_BINS,
            "D_box": dim,
        }
    )
    print(
        f"{key:28s}  D_box={dim:6.3f}  "
        f"occupied={int(occ.sum()):6d}/{FRACTAL_BINS**2}  n={len(x):,}"
    )

fractal_summary = pd.DataFrame(fractal_rows)
age_fractal_summary = pd.DataFrame(age_fractal_rows)
display(fractal_summary)
display(age_fractal_summary)


snaps         D_box= 1.411  occupied=  3176/262144  n=3,242
cascades      D_box= 1.133  occupied=   886/262144  n=892
avalanches    D_box= 0.784  occupied=   225/262144  n=225
JMA           D_box= 1.812  occupied= 51703/262144  n=154,208
Hauksson      D_box= 1.420  occupied=  6564/262144  n=60,943
snaps [10^2, 10^2.5)          D_box= 0.585  occupied=    93/262144  n=94
snaps [10^2.5, 10^3)          D_box= 0.765  occupied=   217/262144  n=217
snaps [10^3, 10^3.5)          D_box= 0.994  occupied=   570/262144  n=573
snaps [10^3.5, 10^4)          D_box= 1.139  occupied=  1083/262144  n=1,099
snaps [10^4, 10^4.5)          D_box= 0.890  occupied=   390/262144  n=391
snaps [10^4.5, 10^5)          D_box= 0.737  occupied=   184/262144  n=187
snaps [10^5, 10^5.5)          D_box= 0.561  occupied=    68/262144  n=68
snaps [10^6, 10^6.5)          D_box= 0.912  occupied=   411/262144  n=417
snaps [10^6.5, 10^7)          D_box= 0.540  occupied=    55/262144  n=55


,catalog,n_events,occupied_pixels,grid,D_box
0,snaps,3242,3176,512,1.410727
1,cascades,892,886,512,1.132588
2,avalanches,225,225,512,0.783556
3,JMA,154208,51703,512,1.811769
4,Hauksson,60943,6564,512,1.420146


,catalog,t_lo,t_hi,n_events,occupied_pixels,grid,D_box
0,"snaps [10^2, 10^2.5)",1.000000e+02,3.162278e+02,94,93,512,0.585168
1,"snaps [10^2.5, 10^3)",3.162278e+02,1.000000e+03,217,217,512,0.765006
2,"snaps [10^3, 10^3.5)",1.000000e+03,3.162278e+03,573,570,512,0.993945
3,"snaps [10^3.5, 10^4)",3.162278e+03,1.000000e+04,1099,1083,512,1.139016
4,"snaps [10^4, 10^4.5)",1.000000e+04,3.162278e+04,391,390,512,0.890278
5,"snaps [10^4.5, 10^5)",3.162278e+04,1.000000e+05,187,184,512,0.736791
6,"snaps [10^5, 10^5.5)",1.000000e+05,3.162278e+05,68,68,512,0.561158
7,"snaps [10^6, 10^6.5)",1.000000e+06,3.162278e+06,417,411,512,0.912215
8,"snaps [10^6.5, 10^7)",3.162278e+06,1.000000e+07,55,55,512,0.539864


In [21]:
fig = go.Figure()
for name, (sizes, counts, dim) in scaling.items():
    if len(sizes) == 0:
        continue
    fig.add_trace(
        go.Scatter(
            x=sizes,
            y=counts,
            mode="lines+markers",
            marker=dict(size=7),
            line=dict(color=COLORS[name], width=2.5),
            name=f"{name} (D={dim:.2f})",
        )
    )

for key, (sizes, counts, dim) in age_scaling.items():
    if len(sizes) == 0:
        continue
    label = key.replace("snaps ", "", 1)
    color = AGE_COLORS.get(label, "#888888")
    fig.add_trace(
        go.Scatter(
            x=sizes,
            y=counts,
            mode="lines+markers",
            marker=dict(size=5, symbol="diamond"),
            line=dict(color=color, width=1.5, dash="dot"),
            name=f"{key} (D={dim:.2f})",
            legendgroup="age",
        )
    )

fig.update_layout(
    title="Box-counting scaling: full catalogs + snaps by Rudyak age window",
    xaxis=dict(title="box size ε (pixels)", type="log"),
    yaxis=dict(title="N(ε)", type="log"),
    height=560,
    margin=dict(l=60, r=20, t=50, b=50),
    legend=dict(orientation="v", yanchor="top", y=1.0, x=1.02),
)
fig.show()


In [22]:
# Row 1: full catalogs. Row 2+: snaps age windows (shared spatial extent).
n_age = len(age_occupancy)
n_cols = max(len(CATALOGS), max(n_age, 1))
n_rows = 1 + (1 if n_age else 0)

titles = [f"{name}<br>D={scaling[name][2]:.2f}" for name in CATALOGS]
titles += [""] * (n_cols - len(CATALOGS))
if n_age:
    age_titles = []
    for key in age_occupancy:
        label = key.replace("snaps ", "", 1)
        dim = age_scaling[key][2]
        age_titles.append(f"snaps {label}<br>D={dim:.2f}")
    age_titles += [""] * (n_cols - len(age_titles))
    titles = titles + age_titles

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=titles,
    horizontal_spacing=0.02,
    vertical_spacing=0.12,
)

for col, name in enumerate(CATALOGS, start=1):
    x, y = xy_coords(name, CATALOGS[name])
    occ = occupancy_grid(x, y, n_bins=FRACTAL_BINS)
    fig.add_trace(
        go.Heatmap(
            z=occ.astype(float),
            showscale=False,
            colorscale=[[0, "#f7f7f7"], [1, COLORS[name]]],
            name=name,
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=1, col=col)
    fig.update_yaxes(
        showticklabels=False,
        scaleanchor=f"x{'' if col == 1 else col}",
        row=1,
        col=col,
    )

for col, key in enumerate(age_occupancy, start=1):
    label = key.replace("snaps ", "", 1)
    color = AGE_COLORS.get(label, "#4C78A8")
    fig.add_trace(
        go.Heatmap(
            z=age_occupancy[key].astype(float),
            showscale=False,
            colorscale=[[0, "#f7f7f7"], [1, color]],
            name=key,
        ),
        row=2,
        col=col,
    )
    fig.update_xaxes(showticklabels=False, row=2, col=col)
    fig.update_yaxes(showticklabels=False, row=2, col=col)

fig.update_layout(
    title_text=(
        f"Spatial occupancy grids ({FRACTAL_BINS}×{FRACTAL_BINS}) "
        "— full catalogs + snaps age windows"
    ),
    height=320 * n_rows + 40,
    margin=dict(l=20, r=20, t=100, b=20),
)
fig.show()


## 9. Nearest-neighbor distances (Zaliapin et al. 2008)

Rescaled nearest-neighbor distances $\eta^\ast$ and the joint $(T, R)$ plane
(Zaliapin–Ben-Zion clustering diagnostics).

Order: full catalogs in time domain → same by Rudyak age window → sims on $t^*=\ln(t)$.


In [23]:
import numba


@numba.njit
def _nn_eta_numba(
    times: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    mags: np.ndarray,
    b: float,
    d: float,
    m0: float,
    q: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """For each event j>0, find parent i<j minimizing n_ij (eq. 1)."""

    n_events = times.shape[0]
    eta = np.empty(n_events, dtype=np.float64)
    t_comp = np.empty(n_events, dtype=np.float64)
    r_comp = np.empty(n_events, dtype=np.float64)
    parent = np.empty(n_events, dtype=np.int64)
    eta[0] = np.nan
    t_comp[0] = np.nan
    r_comp[0] = np.nan
    parent[0] = -1

    for j in range(1, n_events):
        best = 1.0e300
        best_i = -1
        best_tau = 0.0
        best_r = 0.0
        tj = times[j]
        xj = x[j]
        yj = y[j]
        for i in range(j):
            tau = tj - times[i]
            if tau <= 0.0:
                continue
            dx = xj - x[i]
            dy = yj - y[i]
            rij = np.sqrt(dx * dx + dy * dy)
            if rij < 1.0e-12:
                rij = 1.0e-12
            nij = tau * (rij ** d) * (10.0 ** (-b * (mags[i] - m0)))
            if nij < best:
                best = nij
                best_i = i
                best_tau = tau
                best_r = rij
        parent[j] = best_i
        if best_i < 0:
            eta[j] = np.nan
            t_comp[j] = np.nan
            r_comp[j] = np.nan
        else:
            mi = mags[best_i]
            eta[j] = best
            t_comp[j] = best_tau * (10.0 ** (-b * mi * q))
            r_comp[j] = (best_r ** d) * (10.0 ** (-b * mi * (1.0 - q)))
    return eta, t_comp, r_comp, parent


def catalog_xy_km(name: str, frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Spatial coordinates: UTM km for quakes, sheet (x, y) for simulation."""

    if name in {"JMA", "Hauksson"}:
        return (
            frame["x_utm"].to_numpy(dtype=float) / 1000.0,
            frame["y_utm"].to_numpy(dtype=float) / 1000.0,
        )
    return frame["x"].to_numpy(dtype=float), frame["y"].to_numpy(dtype=float)


def catalog_times_years(name: str, frame: pd.DataFrame) -> np.ndarray:
    times = frame["time"].to_numpy(dtype=float)
    if name in {"JMA", "Hauksson"}:
        return times / (365.25 * 86400.0)
    return times


def prepare_nn_frame(
    name: str,
    frame: pd.DataFrame,
    *,
    mag_min: float | None,
    max_events: int,
    seed: int = 0,
) -> pd.DataFrame:
    out = frame.copy()
    mags = magnitude_series(name, out)
    out = out.assign(_m=mags)
    out = out.loc[np.isfinite(out["_m"])].copy()
    if mag_min is not None:
        out = out.loc[out["_m"] >= mag_min].copy()
    out = out.sort_values("time", kind="mergesort").reset_index(drop=True)
    if len(out) > max_events:
        rng = np.random.default_rng(seed)
        keep = np.sort(rng.choice(len(out), size=max_events, replace=False))
        out = out.iloc[keep].reset_index(drop=True)
    return out


def fractal_d_for(name: str, default: float = 1.6) -> float:
    if "fractal_summary" in globals():
        row = fractal_summary.loc[fractal_summary["catalog"] == name]
        if len(row) and np.isfinite(row["D_box"].iloc[0]):
            return float(row["D_box"].iloc[0])
    return default


NN_B = 1.0
NN_Q = 0.5
NN_M0 = 0.0
NN_C = 1.0  # absorbed in η scale; C=1 as in the paper's T,R discussion
NN_MAX_EVENTS = 15_000
NN_MAG_MIN = {
    "snaps": None,
    "cascades": None,
    "avalanches": None,
    "JMA": 4.0,  # Zaliapin–Ben-Zion Japan analyses; keeps η* bimodal
    "Hauksson": 2.0,
}


def load_jma_for_nn() -> pd.DataFrame:
    """JMA events for nearest-neighbor diagnostics (full Japan, notebook time window)."""

    frame = pd.read_csv(
        ROOT / "data/ingested/jma.csv",
        usecols=["time", "magnitude", "latitude", "longitude", "depth", "x_utm", "y_utm"],
    )
    mask = (frame["time"] >= jma_start) & (frame["time"] < jma_end)
    frame = frame.loc[mask].copy()
    frame = frame.sort_values("time", kind="mergesort").reset_index(drop=True)
    frame["dt"] = frame["time"].diff()
    frame["datetime"] = pd.to_datetime(frame["time"], unit="s")
    print(
        f"JMA NN (full Japan, {pd.to_datetime(jma_start, unit='s').date()}–"
        f"{pd.to_datetime(jma_end, unit='s').date()}): {len(frame):,} events before mag cut"
    )
    return frame

# Warm up JIT on a tiny array so later timings exclude compile cost.
_ = _nn_eta_numba(
    np.array([0.0, 1.0]),
    np.array([0.0, 1.0]),
    np.array([0.0, 0.0]),
    np.array([1.0, 1.0]),
    NN_B,
    1.6,
    NN_M0,
    NN_Q,
)
print("numba NN kernel ready")


numba NN kernel ready


### 9.1 · Time domain — compute $\eta^\ast$


In [24]:
def run_nn_catalogs(
    frames: dict[str, pd.DataFrame],
    *,
    label: str,
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """Compute η*, T, R for each catalog in `frames`."""
    results: dict[str, pd.DataFrame] = {}
    meta_rows = []
    for name, frame in frames.items():
        d_frac = fractal_d_for(name)
        prepared = prepare_nn_frame(
            name,
            frame,
            mag_min=NN_MAG_MIN[name],
            max_events=NN_MAX_EVENTS,
            seed=0,
        )
        times = catalog_times_years(name, prepared)
        x_km, y_km = catalog_xy_km(name, prepared)
        mags = prepared["_m"].to_numpy(dtype=float)

        eta, t_comp, r_comp, parent = _nn_eta_numba(
            times, x_km, y_km, mags, NN_B, d_frac, NN_M0, NN_Q
        )
        result = prepared.copy()
        result["eta"] = eta
        result["T"] = t_comp
        result["R"] = r_comp
        result["parent_index"] = parent
        result["log10_eta"] = np.log10(eta)
        result["log10_T"] = np.log10(t_comp)
        result["log10_R"] = np.log10(r_comp)
        results[name] = result

        finite = np.isfinite(eta)
        meta_rows.append(
            {
                "catalog": name,
                "n_used": int(len(result)),
                "n_links": int(finite.sum()),
                "d": d_frac,
                "mag_min": NN_MAG_MIN[name],
                "median_log10_eta": float(np.nanmedian(result["log10_eta"])),
            }
        )
        print(
            f"{label:18s} {name:12s}  n={len(result):5d}  d={d_frac:.3f}  "
            f"mag_min={NN_MAG_MIN[name]}  "
            f"median log10(η*)={np.nanmedian(result['log10_eta']):.2f}"
        )
    return results, pd.DataFrame(meta_rows)


def plot_eta_histograms(
    results: dict[str, pd.DataFrame],
    meta: pd.DataFrame,
    *,
    title: str,
    log_time_names: frozenset[str] | set[str] | None = None,
) -> None:
    log_time_names = log_time_names or frozenset()
    fig = make_subplots(
        rows=len(results),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[
            (
                f"{name} [t*=ln(t)] (n={meta.loc[meta['catalog']==name, 'n_used'].iloc[0]:,})"
                if name in log_time_names
                else f"{name} (n={meta.loc[meta['catalog']==name, 'n_used'].iloc[0]:,})"
            )
            for name in results
        ],
    )
    for row, (name, result) in enumerate(results.items(), start=1):
        vals = result["log10_eta"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) < 2:
            continue
        hist, edges = np.histogram(vals, bins=40, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        fig.add_trace(
            go.Bar(
                x=centers,
                y=hist,
                marker=dict(color=COLORS[name], opacity=0.85),
                name=name,
                showlegend=False,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title_text="density", row=row, col=1)
    fig.update_xaxes(title_text="log₁₀ η*", row=len(results), col=1)
    fig.update_layout(
        title_text=title,
        height=220 * len(results),
        margin=dict(l=60, r=20, t=70, b=50),
        bargap=0.05,
    )
    fig.show()


def plot_tr_joints(
    results: dict[str, pd.DataFrame],
    *,
    title_suffix: str,
    log_time_names: frozenset[str] | set[str] | None = None,
) -> None:
    log_time_names = log_time_names or frozenset()
    for name, result in results.items():
        sample = result.dropna(subset=["log10_T", "log10_R"])
        if len(sample) < 10:
            print(f"skip {name}: too few NN links")
            continue
        if len(sample) > 20_000:
            sample = sample.sample(20_000, random_state=0)

        x = sample["log10_T"].to_numpy(dtype=float)
        y = sample["log10_R"].to_numpy(dtype=float)
        x_range, y_range = tight_limits(x, y)
        med_eta = float(np.nanmedian(result["eta"]))
        tag = f"{name} [t*=ln(t)]" if name in log_time_names else name

        fig = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=(f"{tag} — scatter", f"{tag} — density"),
            horizontal_spacing=0.10,
        )
        fig.add_trace(
            go.Scattergl(
                x=x,
                y=y,
                mode="markers",
                marker=dict(size=4, opacity=0.3, color=COLORS[name]),
                name="NN links",
                showlegend=False,
            ),
            row=1,
            col=1,
        )
        for trace in eta_diagonals(x_range, y_range, (ETA0_PAPER, med_eta)):
            fig.add_trace(trace, row=1, col=1)
        fig.add_trace(
            go.Histogram2d(
                x=x,
                y=y,
                nbinsx=45,
                nbinsy=45,
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title="count", len=0.75, y=0.5),
                name="density",
            ),
            row=1,
            col=2,
        )
        for trace in eta_diagonals(x_range, y_range, (ETA0_PAPER, med_eta)):
            t = go.Scatter(
                x=trace.x,
                y=trace.y,
                mode="lines",
                line=dict(color="white", width=1.5, dash=trace.line.dash),
                name=trace.name,
                showlegend=False,
            )
            fig.add_trace(t, row=1, col=2)
        for col in (1, 2):
            fig.update_xaxes(title_text="log₁₀ T", range=list(x_range), row=1, col=col)
            fig.update_yaxes(title_text="log₁₀ R", range=list(y_range), row=1, col=col)
        fig.update_layout(
            title_text=f"(T, R) — {tag} — {title_suffix} (n={len(sample):,})",
            height=480,
            width=960,
            margin=dict(l=60, r=40, t=60, b=50),
            legend=dict(orientation="h", yanchor="bottom", y=1.08),
        )
        if name == "snaps":
            fig.add_annotation(
                text=(
                    "<b>Note:</b> snaps mag for T,R is "
                    "<b>log₁₀(surface_area)</b> (~constant ≈ 6.52) — "
                    "<b>no real size ladder; mag rescaling is almost uniform</b>"
                ),
                xref="paper",
                yref="paper",
                x=0.5,
                y=-0.14,
                showarrow=False,
                align="center",
                font=dict(size=13, color="#8B0000"),
            )
            fig.update_layout(margin=dict(l=60, r=40, t=60, b=90))
        fig.show()


ETA0_PAPER = 1e-5


def tight_limits(x: np.ndarray, y: np.ndarray, *, pad: float = 0.04):
    """Axis limits from robust data range with a small pad (no equal-aspect lock)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    x0, x1 = np.quantile(x, [0.005, 0.995])
    y0, y1 = np.quantile(y, [0.005, 0.995])
    if not np.isfinite(x0) or x1 <= x0:
        x0, x1 = float(np.min(x)), float(np.max(x))
    if not np.isfinite(y0) or y1 <= y0:
        y0, y1 = float(np.min(y)), float(np.max(y))
    dx = max(x1 - x0, 1e-9)
    dy = max(y1 - y0, 1e-9)
    return (x0 - pad * dx, x1 + pad * dx), (y0 - pad * dy, y1 + pad * dy)


def eta_diagonals(x_range, y_range, eta_values):
    """Line segments log10 R = log10(η) − log10 T clipped to the axis box."""
    traces = []
    x0, x1 = x_range
    y0, y1 = y_range
    styles = {ETA0_PAPER: ("dash", "η₀=1e-5")}
    for eta0 in eta_values:
        if not np.isfinite(eta0) or eta0 <= 0:
            continue
        c = np.log10(eta0)
        xs = np.linspace(x0, x1, 200)
        ys = c - xs
        keep = (ys >= y0) & (ys <= y1)
        if not np.any(keep):
            continue
        dash, label = styles.get(eta0, ("dot", "median η*"))
        traces.append(
            go.Scatter(
                x=xs[keep],
                y=ys[keep],
                mode="lines",
                line=dict(color="black", width=1.5, dash=dash),
                name=label,
                showlegend=True,
            )
        )
    return traces


# JMA nearest-neighbor uses full Japan (not the Ogata box in CATALOGS).
NN_FRAMES = {name: frame for name, frame in CATALOGS.items()}
NN_FRAMES["JMA"] = load_jma_for_nn()

print("=== NN — time domain ===")
nn_results, nn_meta = run_nn_catalogs(NN_FRAMES, label="time domain")
display(nn_meta)


JMA NN (full Japan, 2009-01-01–2014-01-01): 888,516 events before mag cut
=== NN — time domain ===
time domain        snaps         n= 3242  d=1.411  mag_min=None  median log10(η*)=-3.96
time domain        cascades      n=  892  d=1.133  mag_min=None  median log10(η*)=3.58
time domain        avalanches    n=  225  d=0.784  mag_min=None  median log10(η*)=2.81
time domain        JMA           n=12051  d=1.812  mag_min=4.0  median log10(η*)=-6.01
time domain        Hauksson      n=10035  d=1.420  mag_min=2.0  median log10(η*)=-6.83


,catalog,n_used,n_links,d,mag_min,median_log10_eta
0,snaps,3242,3241,1.410727,NaN,-3.959810
1,cascades,892,891,1.132588,NaN,3.575340
2,avalanches,225,224,0.783556,NaN,2.806734
3,JMA,12051,12050,1.811769,4.0,-6.007253
4,Hauksson,10035,10034,1.420146,2.0,-6.828501


### 9.1a · Distribution of $\eta^\ast$ (Zaliapin Figs. 2–3)

Empirical density of $\log_{10}\eta_j^*$ from eq. (1). Clustered catalogs typically show a **bimodal** histogram (background Poisson mode + tight aftershock/cluster mode); a homogeneous Poisson field is unimodal.


In [25]:
plot_eta_histograms(
    nn_results,
    nn_meta,
    title="Nearest-neighbor distance η* — time domain (Zaliapin Figs. 2–3)",
)


### 9.1b · Joint $(T,R)$ (Zaliapin Fig. 4)

From eq. (6) with $q=1/2$: one figure per catalog (scatter + density). Diagonals mark constant $\eta$ ($\eta_0=10^{-5}$ and the catalog median $\eta^*$).


In [26]:
plot_tr_joints(nn_results, title_suffix="time domain")


### 9.2 · By Rudyak age window

Same diagnostics restricted to events whose occurrence time falls in each half-decade age window (§6). Shown for **snaps / cascades / avalanches**. Fractal $d$ uses the per-window snaps estimate when available.


In [27]:
def fractal_d_for_age(label: str, catalog: str, default: float = 1.6) -> float:
    key = f"snaps {label}"
    if "age_fractal_summary" in globals() and len(age_fractal_summary):
        row = age_fractal_summary.loc[age_fractal_summary["catalog"] == key]
        if len(row) and np.isfinite(row["D_box"].iloc[0]):
            return float(row["D_box"].iloc[0])
    return fractal_d_for(catalog, default=default)


def run_nn_age_windows(
    catalogs: dict[str, pd.DataFrame],
    *,
    names: tuple[str, ...] = SIM_NAMES,
    min_events: dict[str, int] | int | None = None,
) -> dict[str, dict[str, object]]:
    """NN results keyed by catalog → age-window label."""
    if min_events is None:
        min_events = {"snaps": 60, "cascades": 60, "avalanches": 25}
    elif isinstance(min_events, int):
        min_events = {name: min_events for name in names}
    out: dict[str, dict[str, object]] = {}
    for name in names:
        frame = catalogs[name]
        out[name] = {}
        n_min = int(min_events.get(name, 60))
        for t_lo, t_hi, label, sub in active_age_windows(frame, min_events=n_min):
            d_frac = fractal_d_for_age(label, name)
            mag_min = NN_MAG_MIN.get(name)
            prepared = prepare_nn_frame(
                name,
                sub,
                mag_min=mag_min,
                max_events=NN_MAX_EVENTS,
                seed=abs(hash((name, label))) % (2**31),
            )
            if len(prepared) < n_min:
                continue
            times = catalog_times_years(name, prepared)
            x, y = catalog_xy_km(name, prepared)
            mags = prepared["_m"].to_numpy(dtype=float)
            eta, t_comp, r_comp, parent = _nn_eta_numba(
                times, x, y, mags, float(NN_B), float(d_frac), float(NN_M0), float(NN_Q)
            )
            result = pd.DataFrame(
                {
                    "eta": eta,
                    "log10_eta": np.log10(eta),
                    "T": t_comp,
                    "R": r_comp,
                    "log10_T": np.log10(t_comp),
                    "log10_R": np.log10(r_comp),
                    "parent": parent,
                }
            )
            out[name][label] = {
                "result": result,
                "d": d_frac,
                "n": len(prepared),
                "mag_min": mag_min,
            }
            med = np.nanmedian(result["log10_eta"].to_numpy(dtype=float))
            print(
                f"{name:12s} t∈{label:22s} n={len(prepared):5d}  "
                f"d={d_frac:.3f}  median log10(η*)={med:.2f}"
            )
    return out


nn_age = run_nn_age_windows(CATALOGS)

# η* histograms: one row per catalog, overlay age windows
for name, windows in nn_age.items():
    if not windows:
        continue
    fig = go.Figure()
    for label, payload in windows.items():
        vals = payload["result"]["log10_eta"].to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) < 5:
            continue
        hist, edges = np.histogram(vals, bins=30, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        fig.add_trace(
            go.Scatter(
                x=centers,
                y=hist,
                mode="lines",
                line=dict(color=AGE_COLORS.get(label, "#888"), width=2),
                name=f"t ∈ {label} (n={payload['n']})",
            )
        )
    fig.update_layout(
        title=f"{name}: η* by Rudyak age window (Zaliapin)",
        xaxis_title="log₁₀ η*",
        yaxis_title="density",
        height=420,
        template="plotly_white",
        legend=dict(orientation="v", yanchor="top", y=1, x=1.02),
        margin=dict(l=60, r=180, t=50, b=50),
    )
    fig.show()

# log T vs log R scatter per age window (small multiples)
for name, windows in nn_age.items():
    labels = [lab for lab, p in windows.items() if p["n"] >= 25]
    if not labels:
        continue
    n = len(labels)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[f"t ∈ {lab}" for lab in labels],
        horizontal_spacing=0.05,
        vertical_spacing=0.12,
    )
    for i, label in enumerate(labels):
        row, col = divmod(i, ncols)
        row += 1
        col += 1
        res = windows[label]["result"]
        x = res["log10_T"].to_numpy(dtype=float)
        y = res["log10_R"].to_numpy(dtype=float)
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        if len(x) == 0:
            continue
        fig.add_trace(
            go.Scattergl(
                x=x,
                y=y,
                mode="markers",
                marker=dict(size=4, color=AGE_COLORS.get(label, "#888"), opacity=0.45),
                name=label,
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        # median η* diagonal: log10 R = log10(η*) - log10 T
        eta_med = np.nanmedian(res["eta"].to_numpy(dtype=float))
        if np.isfinite(eta_med) and eta_med > 0:
            x0, x1 = float(np.nanpercentile(x, 1)), float(np.nanpercentile(x, 99))
            xs = np.linspace(x0, x1, 50)
            ys = np.log10(eta_med) - xs
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=ys,
                    mode="lines",
                    line=dict(color="black", width=1, dash="dot"),
                    showlegend=False,
                ),
                row=row,
                col=col,
            )
        fig.update_xaxes(title_text="log₁₀ T", row=row, col=col)
        fig.update_yaxes(title_text="log₁₀ R", row=row, col=col)
    fig.update_layout(
        title=f"{name}: log₁₀ T vs log₁₀ R by Rudyak age window",
        height=280 * nrows + 60,
        width=980,
        template="plotly_white",
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()


snaps        t∈[10^2, 10^2.5)         n=   94  d=0.585  median log10(η*)=-4.86
snaps        t∈[10^2.5, 10^3)         n=  217  d=0.765  median log10(η*)=-4.30
snaps        t∈[10^3, 10^3.5)         n=  573  d=0.994  median log10(η*)=-4.35
snaps        t∈[10^3.5, 10^4)         n= 1099  d=1.139  median log10(η*)=-4.52
snaps        t∈[10^4, 10^4.5)         n=  391  d=0.890  median log10(η*)=-4.86
snaps        t∈[10^4.5, 10^5)         n=  187  d=0.737  median log10(η*)=-5.06
snaps        t∈[10^5, 10^5.5)         n=   68  d=0.561  median log10(η*)=-5.57
snaps        t∈[10^6, 10^6.5)         n=  417  d=0.912  median log10(η*)=-4.90
cascades     t∈[10^2.5, 10^3)         n=  103  d=0.765  median log10(η*)=2.56
cascades     t∈[10^3, 10^3.5)         n=  211  d=0.994  median log10(η*)=3.21
cascades     t∈[10^3.5, 10^4)         n=  240  d=1.139  median log10(η*)=3.39
cascades     t∈[10^4, 10^4.5)         n=   94  d=0.890  median log10(η*)=3.35
cascades     t∈[10^6, 10^6.5)         n=  101  d=0.912  

### 9.3 · Simulations on $t^* = \ln(t)$


In [28]:
NN_FRAMES_LN = {
    name: (with_log_time(frame) if name in SIM_NAMES else frame)
    for name, frame in NN_FRAMES.items()
}

print("=== NN — simulations on t* = ln(t) ===")
nn_results_ln, nn_meta_ln = run_nn_catalogs(NN_FRAMES_LN, label="t*=ln(t)")
display(nn_meta_ln)


=== NN — simulations on t* = ln(t) ===
t*=ln(t)           snaps         n= 3242  d=1.411  mag_min=None  median log10(η*)=-7.76
t*=ln(t)           cascades      n=  892  d=1.133  mag_min=None  median log10(η*)=0.09
t*=ln(t)           avalanches    n=  225  d=0.784  mag_min=None  median log10(η*)=-0.42
t*=ln(t)           JMA           n=12051  d=1.812  mag_min=4.0  median log10(η*)=-6.01
t*=ln(t)           Hauksson      n=10035  d=1.420  mag_min=2.0  median log10(η*)=-6.83


,catalog,n_used,n_links,d,mag_min,median_log10_eta
0,snaps,3242,3241,1.410727,NaN,-7.760809
1,cascades,892,891,1.132588,NaN,0.092680
2,avalanches,225,224,0.783556,NaN,-0.421670
3,JMA,12051,12050,1.811769,4.0,-6.007253
4,Hauksson,10035,10034,1.420146,2.0,-6.828501


#### Distribution of $\eta^\ast$


In [29]:
plot_eta_histograms(
    nn_results_ln,
    nn_meta_ln,
    title="Nearest-neighbor distance η* — sims on t*=ln(t) (quakes unchanged)",
    log_time_names=frozenset(SIM_NAMES),
)


#### Joint $(T,R)$


In [30]:
plot_tr_joints(
    nn_results_ln,
    title_suffix="sims on t*=ln(t) (quakes unchanged)",
    log_time_names=frozenset(SIM_NAMES),
)


## 10. Yang et al. (2004) first-return-time permutation test

Reproduce Figs. 1–2 of Yang, Du & Ma, *Phys. Rev. Lett.* **92**, 228501 (2004) — *Do Earthquakes Exhibit Self-Organized Criticality?* — for each catalog.

**$P_M(T)$** is the first-return-time density for events with size ≥ M.

| Figure | Catalog |
|---|---|
| **Fig. 1** | original times and magnitudes |
| **Fig. 2** | same occurrence times; magnitudes above completeness randomly permuted (size distribution kept; magnitude–time correlations destroyed) |

Paper details mirrored here:

- log–log row: full waiting-time range, **3 bins per decade**, Omori ∼ T⁻¹
- semi-log row: paper **inset** — same **$P_M(T)$**; x is **$T / T_{\max}(M)$** on [0, 0.1]
- **snaps**: not available — unit events have no usable magnitude ladder

Order: time domain → by Rudyak age window → cascades / avalanches on $t^*=\ln(t)$.


### 10.1 · Time domain


In [31]:
# Yang et al. (2004): first-return-time P_M(T) and magnitude permutation.

YANG_RNG = np.random.default_rng(2004)
YANG_BINS_PER_DECADE = 3
YANG_MIN_RETURNS = 30  # skip M levels with fewer waiting times
YANG_SEMALOG_TMAX_FRAC = 0.1  # paper Fig. 1 inset: T / T_max up to 0.1

# Completeness floor for the permutation (paper: M ≥ 2).
YANG_THRESHOLDS: dict[str, list[float]] = {
    "cascades": [0.0, 0.3, 0.6, 0.9],
    "avalanches": [0.0, 0.5, 1.0, 1.5],
    "JMA": [2.0, 3.0, 4.0, 5.0],
    "Hauksson": [2.0, 3.0, 4.0, 5.0],
}

YANG_SKIP = {
    "snaps": "no usable magnitude (unit events; surface_area nearly degenerate)",
}

YANG_M_COLORS = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#B279A2"]


def first_return_waiting_times(
    times: np.ndarray,
    magnitudes: np.ndarray,
    m_thresh: float,
) -> np.ndarray:
    """Waiting times between successive events with magnitude ≥ m_thresh."""
    times = np.asarray(times, dtype=float)
    magnitudes = np.asarray(magnitudes, dtype=float)
    order = np.argsort(times, kind="mergesort")
    times = times[order]
    magnitudes = magnitudes[order]
    mask = np.isfinite(times) & np.isfinite(magnitudes) & (magnitudes >= m_thresh)
    t_sub = times[mask]
    if len(t_sub) < 3:
        return np.array([], dtype=float)
    dt = np.diff(t_sub)
    return dt[dt > 0]


def pm_density(
    waiting_times: np.ndarray,
    *,
    bins_per_decade: int = YANG_BINS_PER_DECADE,
    linear: bool = False,
    n_linear_bins: int = 40,
    tmax_frac: float = YANG_SEMALOG_TMAX_FRAC,
) -> tuple[np.ndarray, np.ndarray]:
    """Binned P_M(T).

    Log-log: T in seconds, 3 bins/decade over the full range.
    Semi-log (Yang Fig. 1 inset): same P_M density (per second, normalized
    over *all* returns); only the abscissa is T / T_max(M), zoomed to
    [0, tmax_frac].
    """
    wt = np.asarray(waiting_times, dtype=float)
    wt = wt[np.isfinite(wt) & (wt > 0)]
    if len(wt) < 5:
        return np.array([]), np.array([])
    t_min = float(wt.min())
    t_max = float(wt.max())
    if not (t_max > t_min):
        return np.array([]), np.array([])
    n_all = len(wt)
    if linear:
        # Uniform bins in T on [0, tmax_frac * T_max]; density vs T (seconds).
        t_hi = float(tmax_frac) * t_max
        edges_t = np.linspace(0.0, t_hi, n_linear_bins + 1)
        counts, edges_t = np.histogram(wt, bins=edges_t)
        widths_t = np.diff(edges_t)
        centers_t = 0.5 * (edges_t[:-1] + edges_t[1:])
        pdf = counts / (n_all * widths_t)  # same P_M(T) as log-log (per second)
        centers = centers_t / t_max  # only x is normalized
    else:
        n_decades = np.log10(t_max) - np.log10(t_min)
        n_bins = max(int(np.ceil(n_decades * bins_per_decade)), 1)
        edges = np.logspace(np.log10(t_min), np.log10(t_max), n_bins + 1)
        counts, edges = np.histogram(wt, bins=edges)
        widths = np.diff(edges)
        centers = np.sqrt(edges[:-1] * edges[1:])
        pdf = counts / (n_all * widths)
    mask = counts > 0
    return centers[mask], pdf[mask]


def permute_magnitudes_above(
    magnitudes: np.ndarray,
    m_min: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """Shuffle magnitudes among events with m ≥ m_min; leave smaller fixed."""
    mags = np.asarray(magnitudes, dtype=float).copy()
    idx = np.flatnonzero(np.isfinite(mags) & (mags >= m_min))
    if len(idx) < 2:
        return mags
    shuffled = mags[idx].copy()
    rng.shuffle(shuffled)
    mags[idx] = shuffled
    return mags


def yang_thresholds_for(name: str, magnitudes: np.ndarray) -> list[float]:
    if name in YANG_SKIP:
        return []
    mags = np.asarray(magnitudes, dtype=float)
    mags = mags[np.isfinite(mags)]
    preset = YANG_THRESHOLDS.get(name)
    if preset:
        return list(preset)
    lo = float(np.quantile(mags, 0.5))
    hi = float(np.quantile(mags, 0.95))
    if hi <= lo:
        return [lo]
    return list(np.linspace(lo, hi, 4))


def format_thresh(name: str, m: float) -> str:
    if float(m) == int(m):
        return str(int(m))
    return f"{m:g}"


def omori_reference(centers: np.ndarray, pdf: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """P ~ T^{-1} anchored at the mid data point."""
    if len(centers) == 0:
        return np.array([]), np.array([])
    mid = len(centers) // 2
    c0 = float(centers[mid] * pdf[mid])
    t = np.array([centers[0], centers[-1]], dtype=float)
    return t, c0 / t


def yang_pm_curves(
    times: np.ndarray,
    magnitudes: np.ndarray,
    thresholds: list[float],
    *,
    linear: bool = False,
) -> dict[float, tuple[np.ndarray, np.ndarray, int]]:
    out: dict[float, tuple[np.ndarray, np.ndarray, int]] = {}
    for m in thresholds:
        wt = first_return_waiting_times(times, magnitudes, m)
        if len(wt) < YANG_MIN_RETURNS:
            continue
        centers, pdf = pm_density(wt, linear=linear)
        if len(centers) == 0:
            continue
        out[float(m)] = (centers, pdf, int(len(wt)))
    return out


def yang_unavailable_figure(name: str, reason: str) -> go.Figure:
    """Placeholder Fig. 1/2 panel when the catalog has no magnitude ladder."""
    fig = go.Figure()
    fig.add_annotation(
        text=f"<b>{name}: not available</b><br>{reason}",
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=16, color="#555555"),
        align="center",
    )
    fig.update_xaxes(visible=False, range=[0, 1])
    fig.update_yaxes(visible=False, range=[0, 1])
    fig.update_layout(
        title=dict(
            text=f"{name}: Yang et al. Figs. 1–2 — not available",
            x=0.5,
        ),
        height=320,
        width=980,
        template="plotly_white",
        margin=dict(l=40, r=40, t=60, b=40),
    )
    return fig


print("Yang helpers ready.")

# Fig. 1 (original) and Fig. 2 (magnitude-permuted) for every catalog.

yang_results: dict[str, dict] = {}

for name, frame in CATALOGS.items():
    if name in YANG_SKIP:
        reason = YANG_SKIP[name]
        yang_results[name] = {"skipped": True, "reason": reason}
        yang_unavailable_figure(name, reason).show()
        print(f"{name}: not available — {reason}")
        continue

    times = frame["time"].to_numpy(dtype=float)
    mags = magnitude_series(name, frame).to_numpy(dtype=float)
    thresholds = yang_thresholds_for(name, mags)
    if not thresholds:
        print(f"{name}: no thresholds — skip")
        continue
    m_complete = float(min(thresholds))

    mags_perm = permute_magnitudes_above(mags, m_complete, YANG_RNG)

    orig_log = yang_pm_curves(times, mags, thresholds, linear=False)
    perm_log = yang_pm_curves(times, mags_perm, thresholds, linear=False)
    orig_lin = yang_pm_curves(times, mags, thresholds, linear=True)
    perm_lin = yang_pm_curves(times, mags_perm, thresholds, linear=True)

    yang_results[name] = {
        "thresholds": thresholds,
        "m_complete": m_complete,
        "orig_log": orig_log,
        "perm_log": perm_log,
        "orig_lin": orig_lin,
        "perm_lin": perm_lin,
        "n_events": int(np.isfinite(mags).sum()),
        "n_perm_pool": int((np.isfinite(mags) & (mags >= m_complete)).sum()),
    }

    size_label = "log10(n_swaps)" if name in {"cascades", "avalanches"} else "M"

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=False,
        shared_yaxes=False,
        vertical_spacing=0.14,
        horizontal_spacing=0.08,
        subplot_titles=(
            "Fig. 1 original (log–log, full T)",
            "Fig. 2 mag-permuted (log–log, full T)",
            f"Fig. 1 inset (semi-log, T/T_max ≤ {YANG_SEMALOG_TMAX_FRAC:g})",
            f"Fig. 2 inset (semi-log, T/T_max ≤ {YANG_SEMALOG_TMAX_FRAC:g})",
        ),
    )

    panels = [
        (1, 1, orig_log, True),
        (1, 2, perm_log, True),
        (2, 1, orig_lin, False),
        (2, 2, perm_lin, False),
    ]
    for row, col, curves, is_log in panels:
        for i, (m, (centers, pdf, n_wt)) in enumerate(curves.items()):
            color = YANG_M_COLORS[i % len(YANG_M_COLORS)]
            fig.add_trace(
                go.Scatter(
                    x=centers,
                    y=pdf,
                    mode="markers+lines",
                    name=f"{size_label}≥{format_thresh(name, m)} (n={n_wt})",
                    legendgroup=f"m{m:g}",
                    showlegend=(row == 1 and col == 1),
                    marker=dict(size=6, color=color),
                    line=dict(width=1.2, color=color),
                ),
                row=row,
                col=col,
            )
        if is_log and curves:
            m0 = next(iter(curves))
            c0, p0, _ = curves[m0]
            t_ref, p_ref = omori_reference(c0, p0)
            fig.add_trace(
                go.Scatter(
                    x=t_ref,
                    y=p_ref,
                    mode="lines",
                    name="∼ T⁻¹ (Omori)",
                    legendgroup="omori",
                    showlegend=(row == 1 and col == 1),
                    line=dict(width=1.5, color="#333333", dash="dash"),
                ),
                row=row,
                col=col,
            )
        if is_log:
            fig.update_xaxes(type="log", title_text="T (s)", row=row, col=col)
            fig.update_yaxes(type="log", title_text="P_M(T)", row=row, col=col)
        else:
            fig.update_xaxes(
                title_text="T / T_max(M)",
                range=[0.0, YANG_SEMALOG_TMAX_FRAC],
                row=row,
                col=col,
            )
            fig.update_yaxes(type="log", title_text="P_M(T)", row=row, col=col)

    fig.update_layout(
        title=dict(
            text=(
                f"{name}: Yang et al. first-return-time test "
                f"(n={yang_results[name]['n_events']:,}, "
                f"permute pool ≥{format_thresh(name, m_complete)}: "
                f"{yang_results[name]['n_perm_pool']:,})"
            ),
            x=0.5,
        ),
        height=720,
        width=980,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        template="plotly_white",
    )
    fig.show()

    n_orig = {format_thresh(name, m): n for m, (_, _, n) in orig_log.items()}
    n_perm = {format_thresh(name, m): n for m, (_, _, n) in perm_log.items()}
    print(
        f"{name}: thresholds={[format_thresh(name, m) for m in thresholds]}  "
        f"returns orig={n_orig}  perm={n_perm}"
    )
    for m in orig_log:
        if m not in perm_log:
            continue
        wt0 = first_return_waiting_times(times, mags, m)
        wt1 = first_return_waiting_times(times, mags_perm, m)
        t_short = 3600.0 if name in {"JMA", "Hauksson"} else float(np.quantile(wt0, 0.1))
        f0 = float((wt0 < t_short).mean())
        f1 = float((wt1 < t_short).mean())
        tag = format_thresh(name, m)
        print(
            f"  M≥{tag}: P(T<{t_short:g}s) orig={f0:.3f}  perm={f1:.3f}  "
            f"delta={f1 - f0:+.3f}"
        )


Yang helpers ready.


snaps: not available — no usable magnitude (unit events; surface_area nearly degenerate)


cascades: thresholds=['0', '0.3', '0.6', '0.9']  returns orig={'0': 891, '0.3': 497, '0.6': 263, '0.9': 108}  perm={'0': 891, '0.3': 497, '0.6': 263, '0.9': 108}
  M≥0: P(T<1.872s) orig=0.100  perm=0.100  delta=+0.000
  M≥0.3: P(T<3.7035s) orig=0.101  perm=0.117  delta=+0.016
  M≥0.6: P(T<8.2539s) orig=0.103  perm=0.114  delta=+0.011
  M≥0.9: P(T<23.1002s) orig=0.102  perm=0.130  delta=+0.028


avalanches: thresholds=['0', '0.5', '1', '1.5']  returns orig={'0': 224, '0.5': 132, '1': 71}  perm={'0': 224, '0.5': 132, '1': 71}
  M≥0: P(T<6.40485s) orig=0.103  perm=0.103  delta=+0.000
  M≥0.5: P(T<22.4067s) orig=0.106  perm=0.189  delta=+0.083
  M≥1: P(T<54.399s) orig=0.099  perm=0.239  delta=+0.141


JMA: thresholds=['2', '3', '4', '5']  returns orig={'2': 86253, '3': 31153, '4': 6229, '5': 755}  perm={'2': 86253, '3': 31152, '4': 6229, '5': 755}
  M≥2: P(T<3600s) orig=0.872  perm=0.872  delta=+0.000
  M≥3: P(T<3600s) orig=0.751  perm=0.698  delta=-0.053
  M≥4: P(T<3600s) orig=0.605  perm=0.372  delta=-0.233
  M≥5: P(T<3600s) orig=0.506  perm=0.077  delta=-0.429


Hauksson: thresholds=['2', '3', '4', '5']  returns orig={'2': 10025, '3': 1428, '4': 130}  perm={'2': 10025, '3': 1429, '4': 130}
  M≥2: P(T<3600s) orig=0.558  perm=0.558  delta=+0.000
  M≥3: P(T<3600s) orig=0.460  perm=0.321  delta=-0.140
  M≥4: P(T<3600s) orig=0.362  perm=0.062  delta=-0.300


### 10.2 · By Rudyak age window

Yang et al. Fig. 1-style $P_M(T)$ for **cascades** and **avalanches**, restricted to each half-decade age window from §6. Magnitudes and thresholds are unchanged; only the occurrence-time subset varies. Windows with too few returns are skipped.


In [32]:
# Yang first-return P_M(T) per Rudyak age window (cascades / avalanches).

YANG_AGE_NAMES = ("cascades", "avalanches")
YANG_AGE_MIN_EVENTS = {"cascades": 80, "avalanches": 25}


def yang_curves_for_frame(
    name: str,
    frame: pd.DataFrame,
) -> tuple[dict, dict] | None:
    if name in YANG_SKIP:
        return None
    thresholds = YANG_THRESHOLDS.get(name)
    if not thresholds:
        return None
    times = frame["time"].to_numpy(dtype=float)
    mags = magnitude_series(name, frame).to_numpy(dtype=float)
    ok = np.isfinite(times) & np.isfinite(mags)
    times = times[ok]
    mags = mags[ok]
    if len(times) < 25:
        return None
    m_complete = min(thresholds)
    mags_perm = permute_magnitudes_above(mags, m_complete, YANG_RNG)

    def pack(mags_use: np.ndarray) -> dict:
        curves = {}
        for m in thresholds:
            wt = first_return_waiting_times(times, mags_use, m)
            if len(wt) < YANG_MIN_RETURNS:
                continue
            x, y = pm_density(wt, bins_per_decade=YANG_BINS_PER_DECADE, linear=False)
            if len(x):
                curves[m] = (x, y, len(wt))
        return curves

    return pack(mags), pack(mags_perm)


for name in YANG_AGE_NAMES:
    n_min = int(YANG_AGE_MIN_EVENTS[name] if isinstance(YANG_AGE_MIN_EVENTS, dict) else YANG_AGE_MIN_EVENTS)
    windows = active_age_windows(CATALOGS[name], min_events=n_min)
    if not windows:
        print(f"{name}: no age windows with ≥{n_min} events")
        continue
    # One figure per age window: original (left) + permuted (right), log–log
    for t_lo, t_hi, label, sub in windows:
        packed = yang_curves_for_frame(name, sub)
        if packed is None:
            continue
        orig, perm = packed
        if not orig:
            print(f"{name} t∈{label}: insufficient returns")
            continue
        fig = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=("original", "magnitude-permuted"),
            horizontal_spacing=0.08,
        )
        for m_i, m in enumerate(orig):
            color = YANG_M_COLORS[m_i % len(YANG_M_COLORS)]
            x, y, n_ret = orig[m]
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=y,
                    mode="lines+markers",
                    marker=dict(size=5),
                    line=dict(color=color, width=2),
                    name=f"M≥{m:g} (n={n_ret})",
                    legendgroup=f"m{m}",
                ),
                row=1,
                col=1,
            )
            if m in perm:
                xp, yp, n_p = perm[m]
                fig.add_trace(
                    go.Scatter(
                        x=xp,
                        y=yp,
                        mode="lines+markers",
                        marker=dict(size=5),
                        line=dict(color=color, width=2, dash="dash"),
                        name=f"perm M≥{m:g} (n={n_p})",
                        legendgroup=f"m{m}",
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
        # Omori ~ T^{-1} guide on original panel
        if orig:
            x0 = min(v[0].min() for v in orig.values())
            x1 = max(v[0].max() for v in orig.values())
            xs = np.logspace(np.log10(x0), np.log10(x1), 40)
            # normalize guide to pass near first curve mid
            ref_x, ref_y, _ = next(iter(orig.values()))
            mid = len(ref_x) // 2
            scale = ref_y[mid] * ref_x[mid]
            fig.add_trace(
                go.Scatter(
                    x=xs,
                    y=scale / xs,
                    mode="lines",
                    line=dict(color="black", width=1, dash="dot"),
                    name="∼T⁻¹",
                    showlegend=True,
                ),
                row=1,
                col=1,
            )
        fig.update_xaxes(type="log", title_text="T", row=1, col=1)
        fig.update_xaxes(type="log", title_text="T", row=1, col=2)
        fig.update_yaxes(type="log", title_text="P_M(T)", row=1, col=1)
        fig.update_yaxes(type="log", title_text="P_M(T)", row=1, col=2)
        fig.update_layout(
            title=f"{name}: Yang first-return — t ∈ {label} (n={len(sub):,})",
            height=420,
            width=980,
            template="plotly_white",
            legend=dict(orientation="h", yanchor="bottom", y=1.08, x=0),
            margin=dict(l=60, r=20, t=80, b=50),
        )
        fig.show()


avalanches t∈[10^3, 10^3.5): insufficient returns


avalanches t∈[10^4, 10^4.5): insufficient returns


### 10.3 · Cascades / avalanches on $t^* = \ln(t)$

First-return waits are then $\Delta\ln t$. Quakes stay on calendar time. Magnitudes / thresholds are unchanged.


In [33]:
# --- t* = ln(t) for cascades / avalanches ---
YANG_LOG_TIME_NAMES = ("cascades", "avalanches")

for name in YANG_LOG_TIME_NAMES:
    frame = CATALOGS[name]
    times_raw = frame["time"].to_numpy(dtype=float)
    mags_raw = magnitude_series(name, frame).to_numpy(dtype=float)
    keep = np.isfinite(times_raw) & (times_raw > 0) & np.isfinite(mags_raw)
    times = np.log(times_raw[keep])
    mags = mags_raw[keep]
    thresholds = yang_thresholds_for(name, mags)
    if not thresholds:
        print(f"{name} [ln(time)]: no thresholds — skip")
        continue
    m_complete = float(min(thresholds))
    mags_perm = permute_magnitudes_above(mags, m_complete, YANG_RNG)

    orig_log = yang_pm_curves(times, mags, thresholds, linear=False)
    perm_log = yang_pm_curves(times, mags_perm, thresholds, linear=False)
    orig_lin = yang_pm_curves(times, mags, thresholds, linear=True)
    perm_lin = yang_pm_curves(times, mags_perm, thresholds, linear=True)

    key = f"{name}_ln_time"
    yang_results[key] = {
        "thresholds": thresholds,
        "m_complete": m_complete,
        "orig_log": orig_log,
        "perm_log": perm_log,
        "orig_lin": orig_lin,
        "perm_lin": perm_lin,
        "n_events": int(len(mags)),
        "n_perm_pool": int((mags >= m_complete).sum()),
        "time_coord": "ln(time)",
    }

    size_label = "log10(n_swaps)"
    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=False,
        shared_yaxes=False,
        vertical_spacing=0.14,
        horizontal_spacing=0.08,
        subplot_titles=(
            "Fig. 1 original (log–log, full T*)",
            "Fig. 2 mag-permuted (log–log, full T*)",
            f"Fig. 1 inset (semi-log, T*/T*_max ≤ {YANG_SEMALOG_TMAX_FRAC:g})",
            f"Fig. 2 inset (semi-log, T*/T*_max ≤ {YANG_SEMALOG_TMAX_FRAC:g})",
        ),
    )

    panels = [
        (1, 1, orig_log, True),
        (1, 2, perm_log, True),
        (2, 1, orig_lin, False),
        (2, 2, perm_lin, False),
    ]
    for row, col, curves, is_log in panels:
        for i, (m, (centers, pdf, n_wt)) in enumerate(curves.items()):
            color = YANG_M_COLORS[i % len(YANG_M_COLORS)]
            fig.add_trace(
                go.Scatter(
                    x=centers,
                    y=pdf,
                    mode="markers+lines",
                    name=f"{size_label}≥{format_thresh(name, m)} (n={n_wt})",
                    legendgroup=f"m{m:g}",
                    showlegend=(row == 1 and col == 1),
                    marker=dict(size=6, color=color),
                    line=dict(width=1.2, color=color),
                ),
                row=row,
                col=col,
            )
        if is_log and curves:
            m0 = next(iter(curves))
            c0, p0, _ = curves[m0]
            t_ref, p_ref = omori_reference(c0, p0)
            fig.add_trace(
                go.Scatter(
                    x=t_ref,
                    y=p_ref,
                    mode="lines",
                    name="∼ T*⁻¹ (Omori)",
                    legendgroup="omori",
                    showlegend=(row == 1 and col == 1),
                    line=dict(width=1.5, color="#333333", dash="dash"),
                ),
                row=row,
                col=col,
            )
        if is_log:
            fig.update_xaxes(type="log", title_text="T* = Δln t", row=row, col=col)
            fig.update_yaxes(type="log", title_text="P_M(T*)", row=row, col=col)
        else:
            fig.update_xaxes(
                title_text="T* / T*_max(M)",
                range=[0.0, YANG_SEMALOG_TMAX_FRAC],
                row=row,
                col=col,
            )
            fig.update_yaxes(type="log", title_text="P_M(T*)", row=row, col=col)

    fig.update_layout(
        title=dict(
            text=(
                f"{name} [t*=ln(t)]: Yang et al. first-return-time test "
                f"(n={yang_results[key]['n_events']:,}, "
                f"permute pool ≥{format_thresh(name, m_complete)}: "
                f"{yang_results[key]['n_perm_pool']:,})"
            ),
            x=0.5,
        ),
        height=720,
        width=980,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        template="plotly_white",
    )
    fig.show()

    n_orig = {format_thresh(name, m): n for m, (_, _, n) in orig_log.items()}
    n_perm = {format_thresh(name, m): n for m, (_, _, n) in perm_log.items()}
    print(
        f"{name} [ln(time)]: thresholds={[format_thresh(name, m) for m in thresholds]}  "
        f"returns orig={n_orig}  perm={n_perm}"
    )
    for m in orig_log:
        if m not in perm_log:
            continue
        wt0 = first_return_waiting_times(times, mags, m)
        wt1 = first_return_waiting_times(times, mags_perm, m)
        t_short = float(np.quantile(wt0, 0.1))
        f0 = float((wt0 < t_short).mean())
        f1 = float((wt1 < t_short).mean())
        tag = format_thresh(name, m)
        print(
            f"  M≥{tag}: P(T*<{t_short:g}) orig={f0:.3f}  perm={f1:.3f}  "
            f"delta={f1 - f0:+.3f}"
        )


cascades [ln(time)]: thresholds=['0', '0.3', '0.6', '0.9']  returns orig={'0': 891, '0.3': 497, '0.6': 263, '0.9': 108}  perm={'0': 891, '0.3': 497, '0.6': 263, '0.9': 108}
  M≥0: P(T*<0.000206111) orig=0.100  perm=0.100  delta=+0.000
  M≥0.3: P(T*<0.000360673) orig=0.101  perm=0.078  delta=-0.022
  M≥0.6: P(T*<0.000590432) orig=0.103  perm=0.087  delta=-0.015
  M≥0.9: P(T*<0.00184135) orig=0.102  perm=0.093  delta=-0.009


avalanches [ln(time)]: thresholds=['0', '0.5', '1', '1.5']  returns orig={'0': 224, '0.5': 132, '1': 71}  perm={'0': 224, '0.5': 132, '1': 71}
  M≥0: P(T*<0.0147964) orig=0.103  perm=0.103  delta=+0.000
  M≥0.5: P(T*<0.0226169) orig=0.106  perm=0.129  delta=+0.023
  M≥1: P(T*<0.0293006) orig=0.099  perm=0.141  delta=+0.042


## Appendix A · Experimental: fixed-size cascades* / avalanches*

Optional alternate hierarchy levels that force fixed event sizes (experimental; not used by later sections).


### A.1 · Time domain


In [34]:
# Experimental: cascades* = sequential bunches of 10 snaps;
# avalanches* = sequential bunches of 10 cascades*.
BUNCH_SIZE = 10


def _bunch_catalog(
    frame: pd.DataFrame,
    *,
    size: int,
    child_n_swaps: str | None = None,
    child_label: str = "member_ids",
) -> pd.DataFrame:
    """Aggregate every sequential `size` rows (time-sorted) into one event."""
    ordered = frame.sort_values("time", kind="mergesort").reset_index(drop=True)
    n_full = len(ordered) // size
    leftover = len(ordered) - n_full * size
    if n_full == 0:
        raise ValueError(f"need at least {size} events to form one bunch")
    if leftover:
        print(f"  drop {leftover} trailing events (not a full bunch of {size})")
    ordered = ordered.iloc[: n_full * size].copy()
    ordered["_bunch"] = np.arange(len(ordered)) // size

    rows: list[dict] = []
    for bunch_id, group in ordered.groupby("_bunch", sort=True):
        times = group["time"].to_numpy(dtype=float)
        xs = group["x"].to_numpy(dtype=float)
        ys = group["y"].to_numpy(dtype=float)
        zs = group["z"].to_numpy(dtype=float)
        if child_n_swaps is None:
            n_swaps = int(len(group))
        else:
            n_swaps = int(group[child_n_swaps].sum())
        member_ids = tuple(int(v) for v in group["event_id"].tolist())
        row = {
            "event_id": int(bunch_id) + 1,
            "time": float(np.median(times)),
            "time_start": float(times.min()),
            "time_end": float(times.max()),
            "duration": float(times.max() - times.min()),
            "x": float(xs.mean()),
            "y": float(ys.mean()),
            "z": float(zs.mean()),
            "n_swaps": n_swaps,
            "magnitude": float(np.log10(n_swaps)),
            child_label: member_ids,
        }
        if child_n_swaps is not None:
            row["n_cascades"] = int(len(group))
        rows.append(row)
    out = pd.DataFrame(rows)
    out = out.sort_values("time", kind="mergesort").reset_index(drop=True)
    out["dt"] = out["time"].diff()
    out["log10_dt"] = np.nan
    positive = out["dt"] > 0
    out.loc[positive, "log10_dt"] = np.log10(out.loc[positive, "dt"])
    out["cumulative_count"] = np.arange(1, len(out) + 1)
    return out


print("building cascades* from snaps …")
cascades_star = _bunch_catalog(
    bundle.snaps,
    size=BUNCH_SIZE,
    child_n_swaps=None,
    child_label="member_snap_ids",
)
print("building avalanches* from cascades* …")
avalanches_star = _bunch_catalog(
    cascades_star,
    size=BUNCH_SIZE,
    child_n_swaps="n_swaps",
    child_label="member_cascade_ids",
)

CATALOGS_STAR: dict[str, pd.DataFrame] = {
    "snaps": bundle.snaps,
    "cascades*": cascades_star,
    "avalanches*": avalanches_star,
}
CATALOG_STAR_T0 = {name: None for name in CATALOGS_STAR}
COLORS_STAR = {
    "snaps": COLORS["snaps"],
    "cascades*": "#FF9F40",
    "avalanches*": "#C73E3A",
}

print(
    f"snaps={len(bundle.snaps)}  "
    f"cascades*={len(cascades_star)}  "
    f"avalanches*={len(avalanches_star)}"
)
print(
    f"compare real: cascades={len(bundle.cascades)}  "
    f"avalanches={len(bundle.avalanches)}"
)


names_star = list(CATALOGS_STAR)
fig = plot_dt_vs_time(
    CATALOGS_STAR,
    CATALOG_STAR_T0,
    title="dt vs time — experimental fixed-size (10) cascades* / avalanches* (time domain)",
    colors=COLORS_STAR,
    max_points={name: None for name in CATALOGS_STAR},
)
fig.show()


building cascades* from snaps …
  drop 2 trailing events (not a full bunch of 10)
building avalanches* from cascades* …
  drop 4 trailing events (not a full bunch of 10)
snaps=3242  cascades*=324  avalanches*=32
compare real: cascades=892  avalanches=225


### A.2 · Simulations on $t^* = \ln(t)$


In [35]:
CATALOGS_STAR_LN = {
    name: with_log_time(frame) for name, frame in CATALOGS_STAR.items()
}
fig = plot_dt_vs_time(
    CATALOGS_STAR_LN,
    {name: None for name in CATALOGS_STAR_LN},
    title="dt* vs t* — experimental cascades* / avalanches* on t* = ln(t)",
    colors=COLORS_STAR,
    max_points={name: None for name in CATALOGS_STAR_LN},
    x_title="t* − t*_0",
    y_title="dt*",
    log_time_names=frozenset(CATALOGS_STAR_LN),
)
fig.show()
